# 26T3-ENG-R2C1M1P1-CICP-P1: Assessment [Obrigatório]
## Velocidade e Qualidade com Estruturas de Dados e Algoritmos [26E3_1]

---

### Exercício 1: Busca em estruturas lineares e impacto de organização

**Introdução**  
Em sistemas reais, a diferença entre uma busca linear e uma busca logarítmica costuma determinar se uma funcionalidade é utilizável ou não sob carga. Desenvolvedores avançados precisam traduzir requisitos de produto em decisões de engenharia, escolhendo estruturas e algoritmos que entreguem previsibilidade de tempo. Neste exercício, você implementará buscas clássicas e conectará o desempenho à organização dos dados, medindo custos e descrevendo a complexidade em Big O com base em contagens observáveis. Além disso, você fará a ponte entre arrays e estruturas baseadas em nós, evidenciando como a mesma operação pode ter custos radicalmente distintos em listas encadeadas. O objetivo é que você demonstre domínio de implementação, instrumentação e análise, evitando explicações genéricas e fundamentando conclusões em evidências produzidas no notebook.

**Tarefa**  
1. Implemente `linear_search(arr, target)` e `binary_search(sorted_arr, target)` retornando a posição encontrada ou -1.
   - a. Instrumente ambas para contar comparações de chaves e retorne também esse contador;
   - b. Crie um gerador de vetores de tamanho $n$ em três cenários: ordenado, reverso, aleatório;
   - c. Execute para $n$ em pelo menos 5 escalas (por exemplo, $10^2$ a $10^6$) e registre as contagens.
2. Para o caso de `binary_search`, implemente uma etapa explícita de verificação de pré-condição (array ordenado) e defina o comportamento quando a pré-condição falhar.
   - a. Proponha uma estratégia de "falha rápida" com custo menor do que ordenar todo o vetor;
   - b. Justifique a estratégia em termos de Big O e em termos de risco de falso positivo.
3. Implemente uma `SinglyLinkedList` mínima (nó com `value` e `next`) e reescreva a busca linear para operar sobre essa lista.
   - a. Compare as contagens com a busca linear em array para os mesmos dados e explique a diferença prática.

*Tarefa 1: Implementação, Instrumentação e Experimentos*

In [ ]:
import random
import pandas as pd

# 1.a. Algoritmos instrumentados para contagem de comparações
def linear_search(arr, target):
    comparisons = 0
    for i in range(len(arr)):
        comparisons += 1
        if arr[i] == target:
            return i, comparisons
    return -1, comparisons

def binary_search(sorted_arr, target):
    comparisons = 0
    left, right = 0, len(sorted_arr) - 1

    while left <= right:
        mid = (left + right) // 2
        comparisons += 1
        if sorted_arr[mid] == target:
            return mid, comparisons
        elif sorted_arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1, comparisons

# 1.b. Gerador de vetores
def generate_vector(n, scenario):
    if scenario == 'sorted':
        return list(range(n))
    elif scenario == 'reverse':
        return list(range(n, 0, -1))
    elif scenario == 'random':
        vec = list(range(n))
        random.shuffle(vec)
        return vec
    else:
        raise ValueError("Cenário inválido. Escolha 'sorted', 'reverse' ou 'random'.")

# 1.c. Execução experimental em 5 escalas
scales = [10**2, 10**3, 10**4, 10**5, 10**6]
results = []

for n in scales:
    target = -1  # Pior caso (elemento inexistente) para expor o custo assintótico

    for scenario in ['sorted', 'reverse', 'random']:
        arr = generate_vector(n, scenario)

        # Busca Linear
        _, comp_linear = linear_search(arr, target)

        # Busca Binária (válida apenas se ordenado)
        if scenario == 'sorted':
            _, comp_binary = binary_search(arr, target)
        else:
            comp_binary = "N/A"

        results.append({
            'Tamanho (n)': n,
            'Cenário': scenario,
            'Comparações Linear': comp_linear,
            'Comparações Binária': comp_binary
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

 Tamanho (n) Cenário  Comparações Linear Comparações Binária
         100  sorted                 100                   6
         100 reverse                 100                 N/A
         100  random                 100                 N/A
        1000  sorted                1000                   9
        1000 reverse                1000                 N/A
        1000  random                1000                 N/A
       10000  sorted               10000                  13
       10000 reverse               10000                 N/A
       10000  random               10000                 N/A
      100000  sorted              100000                  16
      100000 reverse              100000                 N/A
      100000  random              100000                 N/A
     1000000  sorted             1000000                  19
     1000000 reverse             1000000                 N/A
     1000000  random             1000000                 N/A


*Tarefa 2: Verificação de Pré-Condição e Falha Rápida*

In [ ]:
def is_sorted_fast_fail(arr, k=100):
    """
    Estratégia de Falha Rápida:
    1. Verifica extremidades O(1).
    2. Realiza amostragem de k pares aleatórios adjacentes O(k).
    """
    n = len(arr)
    if n <= 1:
        return True

    # Validação rápida de extremidades
    if arr[0] > arr[-1]:
        return False

    # Validação por amostragem
    sample_size = min(k, n - 1)
    sampled_indices = random.sample(range(n - 1), sample_size)
    for idx in sampled_indices:
        if arr[idx] > arr[idx + 1]:
            return False

    return True

def safe_binary_search(arr, target, k=100):
    if not is_sorted_fast_fail(arr, k):
        raise ValueError("Pré-condição falhou: o array não está devidamente ordenado.")
    return binary_search(arr, target)

*Justificativa Técnica*

- Complexidade Assintótica ($Big\ O$):
  - Verificar se um vetor está totalmente ordenado custa $O(n)$. Ordená-lo antes da busca custa $O(n \log n)$.
  - A estratégia de amostragem proposta possui custo de tempo $O(k)$ em que $k$ é uma constante fixa (ex: $k = 100$). Assintoticamente em relação ao tamanho da entrada $n$, trata-se de um custo $O(1)$, sendo dramaticamente inferior a $O(n)$ e $O(n \log n)$.
- Risco de Falso Positivo:
  - Por ser uma verificação probabilística e parcial, há o risco de o algoritmo declarar um vetor como ordenado quando ele apresenta inversões locais fora dos pontos amostrados.
  - Caso ocorra um falso positivo, a busca binária executará sobre dados desalinhados, podendo retornar $-1$ mesmo se o elemento estiver presente ou retornar um índice incorreto.

*Tarefa 3: SinglyLinkedList e Comparativo Prático*

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class SinglyLinkedList:
    def __init__(self):
        self.head = None

    def append(self, value):
        new_node = Node(value)
        if not self.head:
            self.head = new_node
            return
        curr = self.head
        while curr.next:
            curr = curr.next
        curr.next = new_node

def linear_search_sll(sll, target):
    comparisons = 0
    curr = sll.head
    idx = 0
    while curr:
        comparisons += 1
        if curr.value == target:
            return idx, comparisons
        curr = curr.next
        idx += 1
    return -1, comparisons

# Experimento comparativo
n = 10**4
data = generate_vector(n, 'random')

sll = SinglyLinkedList()
for val in data:
    sll.append(val)

_, comp_array = linear_search(data, -1)
_, comp_sll = linear_search_sll(sll, -1)

print(f"Comparações no Array: {comp_array}")
print(f"Comparações na SinglyLinkedList: {comp_sll}")

Comparações no Array: 10000
Comparações na SinglyLinkedList: 10000


*Comparação e Explicação Prática*
- Contagem de Comparações de Chaves: É identicamente igual entre o Array e a SinglyLinkedList para uma mesma sequência de dados. Em ambos os casos, no pior cenário de busca linear, o algoritmo realiza exatamente $n$ comparações ($O(n)$).
- Diferença Prática de Desempenho (Tempo Real de CPU):
  - Array (Lista em Python): Armazena referências de forma contígua em memória. Isso favorece a localidade espacial de referência, permitindo que a CPU utilize mecanismos de prefetching e aproveite os caches L1/L2 com alto índice de cache hits.
  - SinglyLinkedList: Cada nó é instanciado em posições esparsas da memória Heap. A navegação exige um salto de ponteiro a cada elemento (curr = curr.next), gerando frequentes cache misses e maior sobrecarrega (overhead) de memória devido à alocação de ponteiros. Na prática, o tempo de execução do tempo de CPU na lista encadeada é significativamente superior ao do array, mesmo com o mesmo número de comparações de chaves.

### Exercício 2: Ordenações quadráticas com instrumentação e diagnóstico

**Introdução**  
Ordenar dados é uma operação central em engenharia de software, porém o ponto crítico não é "saber ordenar", e sim reconhecer quando uma implementação é inadequada para o volume e o padrão de entrada. Algoritmos como Bubble Sort, Selection Sort e Insertion Sort têm complexidades conhecidas, mas um engenheiro precisa conseguir demonstrar esse custo com medições internas do algoritmo, como número de cópias e comparações. Neste exercício, você implementará essas ordenações com instrumentação rigorosa, testará em diferentes distribuições e produzirá uma análise que conecte comportamento observado e Big O. Ao final, você integrará a ordenação com uma estrutura em árvore para validar invariantes de ordenação por meio de travessias, reforçando o vínculo entre arrays e estruturas dinâmicas.

**Tarefa**  
1. Implemente `bubble_sort`, `selection_sort` e `insertion_sort` sobre listas Python, retornando também:
   - a. número de comparações de chaves;
   - b. número de cópias (considere uma troca como 3 cópias).
2. Projete um experimento reprodutível com pelo menos quatro padrões de entrada: ordenado, reverso, quase ordenado, aleatório.
   - a. Para cada padrão, execute com ao menos 4 tamanhos crescentes e registre comparações e cópias;
   - b. Identifique qual algoritmo se beneficia mais do caso "quase ordenado" e explique por que isso ocorre.
3. Construa uma BinarySearch Tree (BST) de inteiros e use-a para validar ordenação.
   - a. Insira todos os elementos do array original na BST, faça uma travessia in-order e verifique que o resultado é ordenado;
   - b. Compare o custo total de ordenação por BST (inserções + in-order) com Insertion Sort, usando Big O e evidências do seu experimento.

*Tarefa 1: Algoritmos de Ordenação Instrumentados*

In [ ]:
def bubble_sort(arr):
    arr_copy = arr.copy()
    n = len(arr_copy)
    comparisons = 0
    copies = 0

    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            comparisons += 1
            if arr_copy[j] > arr_copy[j + 1]:
                # Troca (swap) equivale a 3 cópias de elementos
                arr_copy[j], arr_copy[j + 1] = arr_copy[j + 1], arr_copy[j]
                copies += 3
                swapped = True
        if not swapped:
            break

    return arr_copy, comparisons, copies


def selection_sort(arr):
    arr_copy = arr.copy()
    n = len(arr_copy)
    comparisons = 0
    copies = 0

    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            comparisons += 1
            if arr_copy[j] < arr_copy[min_idx]:
                min_idx = j

        if min_idx != i:
            # Troca entre o elemento atual e o mínimo encontrado (3 cópias)
            arr_copy[i], arr_copy[min_idx] = arr_copy[min_idx], arr_copy[i]
            copies += 3

    return arr_copy, comparisons, copies


def insertion_sort(arr):
    arr_copy = arr.copy()
    n = len(arr_copy)
    comparisons = 0
    copies = 0

    for i in range(1, n):
        key = arr_copy[i]
        copies += 1  # Guarda a chave em variável temporária
        j = i - 1

        while j >= 0:
            comparisons += 1
            if arr_copy[j] > key:
                arr_copy[j + 1] = arr_copy[j]
                copies += 1  # Deslocamento de elemento
                j -= 1
            else:
                break

        arr_copy[j + 1] = key
        copies += 1  # Inserção da chave na posição correta

    return arr_copy, comparisons, copies

*Tarefa 2: Experimento Reprodutível e Análise*

In [ ]:
import random
import pandas as pd

def generate_data(n, pattern):
    if pattern == 'sorted':
        return list(range(n))
    elif pattern == 'reverse':
        return list(range(n, 0, -1))
    elif pattern == 'nearly_sorted':
        arr = list(range(n))
        # Provoca trocas em aproximadamente 5% dos elementos
        swaps = max(1, int(n * 0.05))
        for _ in range(swaps):
            i, j = random.sample(range(n), 2)
            arr[i], arr[j] = arr[j], arr[i]
        return arr
    elif pattern == 'random':
        arr = list(range(n))
        random.shuffle(arr)
        return arr
    else:
        raise ValueError("Padrão inválido.")

sizes = [100, 300, 500, 1000]
patterns = ['sorted', 'reverse', 'nearly_sorted', 'random']
exp_results = []

for size in sizes:
    for pat in patterns:
        data = generate_data(size, pat)

        _, comp_b, copy_b = bubble_sort(data)
        _, comp_s, copy_s = selection_sort(data)
        _, comp_i, copy_i = insertion_sort(data)

        exp_results.append({
            'Tamanho (N)': size,
            'Padrão': pat,
            'Bubble (Comp/Cóp)': f"{comp_b} / {copy_b}",
            'Selection (Comp/Cóp)': f"{comp_s} / {copy_s}",
            'Insertion (Comp/Cóp)': f"{comp_i} / {copy_i}"
        })

df_exp = pd.DataFrame(exp_results)
print(df_exp.to_string(index=False))

 Tamanho (N)        Padrão Bubble (Comp/Cóp) Selection (Comp/Cóp) Insertion (Comp/Cóp)
         100        sorted            99 / 0             4950 / 0             99 / 198
         100       reverse      4950 / 14850           4950 / 150          4950 / 5148
         100 nearly_sorted        4355 / 993            4950 / 15            430 / 529
         100        random       4905 / 7227           4950 / 297          2504 / 2607
         300        sorted           299 / 0            44850 / 0            299 / 598
         300       reverse    44850 / 134550          44850 / 450        44850 / 45448
         300 nearly_sorted      41195 / 7005           44850 / 45          2634 / 2933
         300        random     44679 / 65580          44850 / 882        22152 / 22458
         500        sorted           499 / 0           124750 / 0            499 / 998
         500       reverse   124750 / 374250         124750 / 750      124750 / 125748
         500 nearly_sorted    122605 / 2822

*Análise do Caso "Quase Ordenado*
- Algoritmo Beneficiado: O Insertion Sort.   
- Justificação Técnica:
  - No Insertion Sort, o laço interno while interrompe a verificação assim que encontra um elemento menor ou igual à chave. Quando o vetor está "quase ordenado", a maioria dos elementos já se encontra próxima da sua posição final.
  - Isso reduz drasticamente o número de comparações e deslocamentos internos por elemento para $O(1)$ no caso ideal, aproximando o custo total de tempo de $O(n)$.   
  - Em contrapartida, o Selection Sort realiza sempre $\frac{n(n-1)}{2}$ comparações ($O(n^2)$) independentemente da ordenação prévia. O Bubble Sort otimizado deteta a ausência de trocas, mas ainda assim efetua mais comparações do que o Insertion Sort para pequenas perturbações locais.

*Tarefa 3: Ordenação e Validação via Binary Search Tree (BST)*

In [ ]:
class BSTNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None
        self.insert_ops = 0  # Contabilizador de comparações na inserção

    def insert(self, key):
        if not self.root:
            self.root = BSTNode(key)
        else:
            self._insert_recursive(self.root, key)

    def _insert_recursive(self, node, key):
        self.insert_ops += 1
        if key < node.key:
            if node.left is None:
                node.left = BSTNode(key)
            else:
                self._insert_recursive(node.left, key)
        else:
            if node.right is None:
                node.right = BSTNode(key)
            else:
                self._insert_recursive(node.right, key)

    def inorder(self):
        result = []
        self._inorder_recursive(self.root, result)
        return result

    def _inorder_recursive(self, node, result):
        if node:
            self._inorder_recursive(node.left, result)
            result.append(node.key)
            self._inorder_recursive(node.right, result)


# Validação e Experimento Comparativo com Insertion Sort
test_data = generate_data(500, 'random')

# 3.a. Inserção na BST e verificação via travessia in-order
bst = BST()
for val in test_data:
    bst.insert(val)

bst_sorted_data = bst.inorder()
is_valid = bst_sorted_data == sorted(test_data)

# Execução do Insertion Sort para dados idênticos
_, ins_comp, ins_copies = insertion_sort(test_data)

print(f"Resultado da BST está ordenado? {is_valid}")
print(f"Operações de inserção (comparações) na BST: {bst.insert_ops}")
print(f"Comparações no Insertion Sort: {ins_comp}")

Resultado da BST está ordenado? True
Operações de inserção (comparações) na BST: 4599
Comparações no Insertion Sort: 63888


*Comparação de Custo Assintótico e Prático (BST vs Insertion Sort)*
- Custo da BST (Inserções + Travessia In-order):
  - Caso Médio (Entrada Aleatória): A inserção de $n$ elementos numa BST não balanceada tem custo de tempo $O(n \log n)$. A travessia in-order posterior percorre cada nó exatamente uma vez em $O(n)$. O custo total resultante é $O(n \log n)$.   
  - Pior Caso (Entrada Ordenada ou Reversa): A BST degrada numa lista encadeada (árvore degenerada) com altura $h = n$. O custo de inserção eleva-se para $O(n^2)$, resultando num custo global de $O(n^2)$.
- Insertion Sort:
  - Apresenta custo de tempo $O(n)$ para o melhor caso (ordenado) e $O(n^2)$ para casos médios e piores (aleatório/reverso).  
- Conclusão com Base em Evidências: Em vetores aleatórios de média/grande escala, o método BST supera o Insertion Sort por atingir complexidade $O(n \log n)$ contra o custo quadrático $O(n^2)$ do Insertion Sort. No entanto, em vetores já ordenados, o Insertion Sort é superior ($O(n)$) enquanto a BST sofre degradação de desempenho ($O(n^2)$).



### Exercício 3: Otimização orientada a Big O com requisitos de desempenho

**Introdução**  
Otimizar não é "tornar mais rápido" de forma vaga, é reduzir o custo assintótico e os fatores dominantes sob restrições explícitas. Em ambientes profissionais, você receberá um código funcional que falha em escala, e sua responsabilidade será redesenhá-lo com base em análise e instrumentação. Neste exercício, você partirá de uma solução deliberadamente ineficiente para deduplicação e seleção de estatísticas, reescrevendo-a para reduzir complexidade e justificar tecnicamente cada decisão. O exercício exige que você demonstre capacidade de reconhecer gargalos, propor estruturas adequadas e validar corretude com testes. Para consolidar a integração com estruturas dinâmicas, você deverá suportar uma variante em árvore, comparando a eficiência de operações e evidenciando trade-offs de implementação.

**Tarefa**  
1. Implemente `deduplicate_slow(arr)` com abordagem ingênua $O(n^2)$ e depois implemente `deduplicate_fast(arr)` com $O(n)$.
   - a. Em `deduplicate_fast`, você deve preservar ordem relativa dos primeiros aparecimentos;
   - b. Instrumente ambas para contar comparações e escreva testes que provem equivalência de saída.
2. Implemente `k_smallest(arr, k)` de duas formas:
   - a. versão A: ordena todo o array e pega os $k$ primeiros;
   - b. versão B: usa QuickSelect para encontrar o $k$-ésimo e particionar sem ordenar totalmente;
   - c. Compare as duas em termos de Big O e em contagens de operações para diferentes tamanhos e valores de $k$.
3. Integre uma variante em árvore:
   - a. Insira os valores em uma BST e implemente `k_smallest_bst(root, k)` por travessia controlada;
   - b. Meça a eficiência esperada e discuta o pior caso, incluindo quando a BST degrada para lista encadeada.

*Tarefa 1: Deduplicação Ineficiente ($O(n^2)$) e Otimizada ($O(n)$)*

In [ ]:
import random

# 1.a & 1.b: Abordagem $O(n^2)$ por pesquisa linear na lista de saída
def deduplicate_slow(arr):
    seen = []
    comparisons = 0
    for item in arr:
        found = False
        for x in seen:
            comparisons += 1
            if x == item:
                found = True
                break
        if not found:
            seen.append(item)
    return seen, comparisons

# 1.a & 1.b: Abordagem $O(n)$ preservando a ordem com Tabela de Dispersão (Set)
def deduplicate_fast(arr):
    seen_set = set()
    result = []
    comparisons = 0
    for item in arr:
        comparisons += 1 # Verificação/Hash lookup no conjunto
        if item not in seen_set:
            seen_set.add(item)
            result.append(item)
    return result, comparisons

# Testes automatizados para comprovar equivalência e comparar operações
def test_deduplication():
    test_cases = [
        [1, 2, 2, 3, 4, 4, 4, 5],
        [5, 4, 3, 2, 1],
        [1, 1, 1, 1, 1],
        [random.randint(1, 50) for _ in range(1000)]
    ]

    for idx, arr in enumerate(test_cases, 1):
        slow_res, slow_comp = deduplicate_slow(arr)
        fast_res, fast_comp = deduplicate_fast(arr)

        # Validação de equivalência
        assert slow_res == fast_res, f"Falha de equivalência no caso {idx}"

        print(f"Caso {idx} (N={len(arr)}): Resultados idênticos.")
        print(f"  - Lento O(n²): {slow_comp} comparações")
        print(f"  - Rápido O(n): {fast_comp} comparações\n")

test_deduplication()

Caso 1 (N=8): Resultados idênticos.
  - Lento O(n²): 20 comparações
  - Rápido O(n): 8 comparações

Caso 2 (N=5): Resultados idênticos.
  - Lento O(n²): 10 comparações
  - Rápido O(n): 5 comparações

Caso 3 (N=5): Resultados idênticos.
  - Lento O(n²): 4 comparações
  - Rápido O(n): 5 comparações

Caso 4 (N=1000): Resultados idênticos.
  - Lento O(n²): 24416 comparações
  - Rápido O(n): 1000 comparações



*Tarefa 2: Seleção dos $k$ Menores Elementos (k_smallest)*

In [ ]:
# 2.a: Versão A - Ordenação Total O(n log n)
def k_smallest_sort(arr, k):
    arr_copy = arr.copy()
    comparisons = 0

    # Simulação de ordenação com contagem (usando Python sorted / Timsort)
    # Timsort faz no máximo n * log2(n) comparações no caso médio/pior
    import math
    n = len(arr_copy)
    comparisons = int(n * math.log2(n)) if n > 0 else 0

    sorted_arr = sorted(arr_copy)
    return sorted_arr[:k], comparisons

# 2.b: Versão B - QuickSelect O(n) médio
def quickselect_partition(arr, low, high, stats):
    pivot = arr[high]
    i = low - 1
    for j in range(low, high):
        stats['comparisons'] += 1
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
            stats['swaps'] += 1
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    stats['swaps'] += 1
    return i + 1

def quickselect_helper(arr, low, high, k_idx, stats):
    if low <= high:
        pivot_idx = quickselect_partition(arr, low, high, stats)
        if pivot_idx == k_idx:
            return
        elif pivot_idx > k_idx:
            quickselect_helper(arr, low, pivot_idx - 1, k_idx, stats)
        else:
            quickselect_helper(arr, pivot_idx + 1, high, k_idx, stats)

def k_smallest_quickselect(arr, k):
    arr_copy = arr.copy()
    stats = {'comparisons': 0, 'swaps': 0}
    if k > 0 and k <= len(arr_copy):
        quickselect_helper(arr_copy, 0, len(arr_copy) - 1, k - 1, stats)
    return sorted(arr_copy[:k]), stats['comparisons'] # Retorna os k menores particionados

# 2.c: Experimento comparativo entre Versão A e Versão B
import pandas as pd

sizes = [500, 2000, 10000]
k_ratios = [0.05, 0.25, 0.50]
k_results = []

for n in sizes:
    data = [random.randint(1, 100000) for _ in range(n)]
    for ratio in k_ratios:
        k = int(n * ratio)
        _, comp_a = k_smallest_sort(data, k)
        _, comp_b = k_smallest_quickselect(data, k)

        k_results.append({
            'Tamanho (N)': n,
            'k': k,
            'Proporção k/N': f"{int(ratio*100)}%",
            'Versão A (Sort) - Comparações Est.': comp_a,
            'Versão B (QuickSelect) - Comparações': comp_b
        })

df_k = pd.DataFrame(k_results)
print(df_k.to_string(index=False))

 Tamanho (N)    k Proporção k/N  Versão A (Sort) - Comparações Est.  Versão B (QuickSelect) - Comparações
         500   25            5%                                4482                                  1501
         500  125           25%                                4482                                  2360
         500  250           50%                                4482                                  2508
        2000  100            5%                               21931                                  3542
        2000  500           25%                               21931                                  5528
        2000 1000           50%                               21931                                  3910
       10000  500            5%                              132877                                 24838
       10000 2500           25%                              132877                                 22229
       10000 5000           50%               

*Comparativo de Complexidade ($Big\ O$)*
- Versão A (Ordenação Total): Possui complexidade temporal $O(n \log n)$. Ordena todo o vetor independentemente do valor de $k$, realizando trabalho desnecessário para $k \ll n$.   
- Versão B (QuickSelect): Possui complexidade temporal média de $O(n)$. Descarta recursivamente uma das partições, reduzindo a área de busca a cada passo ($\approx n + \frac{n}{2} + \frac{n}{4} + \dots = 2n$). No entanto, o seu pior caso assintótico é $O(n^2)$ se os pivôs forem mal escolhidos.

*Tarefa 3: Seleção de $k$-Menores em BST e Análise*

In [ ]:
class BSTNode:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def insert(self, val):
        if not self.root:
            self.root = BSTNode(val)
        else:
            self._insert_rec(self.root, val)

    def _insert_rec(self, node, val):
        if val < node.val:
            if node.left is None:
                node.left = BSTNode(val)
            else:
                self._insert_rec(node.left, val)
        else:
            if node.right is None:
                node.right = BSTNode(val)
            else:
                self._insert_rec(node.right, val)

# 3.a: Travessia in-order interrompida ao atingir k elementos
def k_smallest_bst(root, k):
    result = []
    visited_nodes = 0

    def inorder(node):
        nonlocal visited_nodes
        if not node or len(result) == k:
            return

        inorder(node.left)

        if len(result) < k:
            result.append(node.val)
            visited_nodes += 1
            inorder(node.right)

    inorder(root)
    return result, visited_nodes

# Exemplo de teste com dados aleatórios
data_tree = [random.randint(1, 1000) for _ in range(100)]
bst = BST()
for x in data_tree:
    bst.insert(x)

k_val = 10
res_bst, nodes_visited = k_smallest_bst(bst.root, k_val)
print(f"Os {k_val} menores elementos via BST: {res_bst}")
print(f"Nós visitados na travessia controlada: {nodes_visited}")

Os 10 menores elementos via BST: [6, 18, 20, 31, 33, 34, 41, 42, 63, 63]
Nós visitados na travessia controlada: 10


*Análise de Eficiência e Degradação da BST*
- Eficiência Esperada:
  - Construção da Árvore: Para $n$ elementos inseridos aleatoriamente, a altura esperada é $O(\log n)$, custando $O(n \log n)$ no total.  
  - Consulta dos $k$ Menores: A travessia in-order interrompida percorre apenas a subárvore esquerda relevante e atinge os $k$ primeiros nós em tempo $O(h + k)$, em que $h$ é a altura da árvore.
- Eficiência Esperada:
  - Se os dados forem inseridos de forma estritamente ordenada ou inversa, a BST degrada numa lista encadeada (árvore desbalanceada de altura $h = n$).
  - Na BST degenerada, a inserção atinge o custo quadrático $O(n^2)$. A busca pelos $k$ menores passa a exigir a travessia sequencial de $k$ nós profundos com custo $O(k)$ sobre uma estrutura linear de acesso $O(n)$.     


### Exercício 4: Hashtable com colisões e encadeamento por lista

**Introdução**  
Hashtables são a base de índices, caches, memoization e inúmeras estruturas de alto desempenho. O ponto de maturidade técnica é entender que "$O(1)$" é uma promessa condicional, e que colisões, fator de carga e estratégia de resolução determinam o desempenho real. Neste exercício, você implementará uma hashtable do zero, com hashing, redimensionamento e tratamento explícito de colisões. A exigência é que a colisão seja resolvida por encadeamento, usando uma lista encadeada por bucket, o que conecta diretamente estruturas baseadas em nós com acesso eficiente. Você também deverá fornecer uma análise de complexidade dependente do fator de carga e demonstrar, com testes, que a estrutura mantém corretude após rehash. A qualidade da entrega será avaliada pela precisão da implementação e pela clareza das evidências.

**Tarefa**  
1. Implemente `HashTableChained` com métodos `put(key, value)`, `get(key)`, `delete(key)` e `__len__()`.
   - a. Cada bucket deve ser uma lista encadeada de pares `(key, value)`;
   - b. Implemente atualização de valor para chave existente sem inserir duplicata.
2. Implemente redimensionamento automático baseado em fator de carga.
   - a. Defina um limiar e faça rehash para uma capacidade maior;
   - b. Demonstre por teste que todas as chaves permanecem acessíveis após o rehash.
3. Instrumente e avalie desempenho.
   - a. Conte comparações dentro dos buckets e relacione o custo esperado com o fator de carga;
   - b. Construa um cenário adversarial com chaves que colidem e discuta o impacto em Big O.

*Tarefa 1 e Tarefa 2: Implementação da HashTableChained com Redimensionamento Automático*

In [ ]:
class HashNode:
    def __init__(self, key, value):
        self.key = key
        self.value = value
        self.next = None

class HashTableChained:
    def __init__(self, initial_capacity=8, load_factor_threshold=0.75):
        self.capacity = initial_capacity
        self.load_factor_threshold = load_factor_threshold
        self.buckets = [None] * self.capacity
        self.size = 0
        self.last_op_comparisons = 0

    def _hash(self, key):
        # Função de hashing utilizando o hash embutido adaptado à capacidade atual
        return hash(key) % self.capacity

    def __len__(self):
        return self.size

    def get_load_factor(self):
        return self.size / self.capacity

    def put(self, key, value):
        index = self._hash(key)
        curr = self.buckets[index]

        # 1.b: Atualização de valor para chave existente sem inserir duplicata
        while curr:
            if curr.key == key:
                curr.value = value
                return
            curr = curr.next

        # Inserção no início da lista encadeada do bucket
        new_node = HashNode(key, value)
        new_node.next = self.buckets[index]
        self.buckets[index] = new_node
        self.size += 1

        # 2.a: Redimensionamento automático baseado no fator de carga
        if self.get_load_factor() > self.load_factor_threshold:
            self._resize(self.capacity * 2)

    def get(self, key):
        index = self._hash(key)
        curr = self.buckets[index]
        self.last_op_comparisons = 0

        while curr:
            self.last_op_comparisons += 1
            if curr.key == key:
                return curr.value
            curr = curr.next
        return None

    def delete(self, key):
        index = self._hash(key)
        curr = self.buckets[index]
        prev = None
        self.last_op_comparisons = 0

        while curr:
            self.last_op_comparisons += 1
            if curr.key == key:
                if prev:
                    prev.next = curr.next
                else:
                    self.buckets[index] = curr.next
                self.size -= 1
                return True
            prev = curr
            curr = curr.next
        return False

    def _resize(self, new_capacity):
        old_buckets = self.buckets
        self.capacity = new_capacity
        self.buckets = [None] * self.capacity
        self.size = 0

        # Rehash de todos os elementos para a nova estrutura
        for head in old_buckets:
            curr = head
            while curr:
                # Reinsere cada elemento ajustando para o novo índice
                index = self._hash(curr.key)
                new_node = HashNode(curr.key, curr.value)
                new_node.next = self.buckets[index]
                self.buckets[index] = new_node
                self.size += 1
                curr = curr.next


# 2.b: Teste que demonstra acessibilidade das chaves após Rehash
def test_rehash():
    ht = HashTableChained(initial_capacity=4, load_factor_threshold=0.75)
    test_data = {f"key_{i}": i * 10 for i in range(20)}

    # Inserção de dados provocando múltiplos rehashes
    initial_cap = ht.capacity
    for k, v in test_data.items():
        ht.put(k, v)

    print(f"Capacidade inicial: {initial_cap}")
    print(f"Capacidade final pós-rehash: {ht.capacity}")
    print(f"Fator de Carga final: {ht.get_load_factor():.2f}")

    # Verificação de acessibilidade
    all_accessible = all(ht.get(k) == v for k, v in test_data.items())
    print(f"Todas as {len(test_data)} chaves continuam acessíveis? {all_accessible}\n")

test_rehash()

Capacidade inicial: 4
Capacidade final pós-rehash: 32
Fator de Carga final: 0.62
Todas as 20 chaves continuam acessíveis? True



*Tarefa 3: Instrumentação e Avaliação de Desempenho*

In [ ]:
class AdversarialKey:
    """Chave customizada para forçar colisões no mesmo bucket."""
    def __init__(self, val):
        self.val = val

    def __hash__(self):
        return 42  # Força todos os elementos a colidirem no mesmo hash

    def __eq__(self, other):
        return isinstance(other, AdversarialKey) and self.val == other.val

    def __repr__(self):
        return f"AdvKey({self.val})"


# 3.a & 3.b: Experimento Comparativo (Cenário Normal vs Cenário Adversarial)
def evaluate_hashtable_performance():
    n = 500

    # 1. Cenário Normal (Chaves distribuídas)
    ht_normal = HashTableChained(initial_capacity=16, load_factor_threshold=0.75)
    for i in range(n):
        ht_normal.put(f"normal_key_{i}", i)

    normal_comparisons = []
    for i in range(n):
        ht_normal.get(f"normal_key_{i}")
        normal_comparisons.append(ht_normal.last_op_comparisons)

    avg_comp_normal = sum(normal_comparisons) / len(normal_comparisons)

    # 2. Cenário Adversarial (Todas as chaves colidem no mesmo bucket)
    ht_adversarial = HashTableChained(initial_capacity=16, load_factor_threshold=0.75)
    for i in range(n):
        ht_adversarial.put(AdversarialKey(i), i)

    adv_comparisons = []
    for i in range(n):
        ht_adversarial.get(AdversarialKey(i))
        adv_comparisons.append(ht_adversarial.last_op_comparisons)

    avg_comp_adv = sum(adv_comparisons) / len(adv_comparisons)

    print("--- Resultados de Avaliação de Desempenho ---")
    print(f"Cenário Normal   - Média de Comparações/Busca: {avg_comp_normal:.2f}")
    print(f"Cenário Adversarial - Média de Comparações/Busca: {avg_comp_adv:.2f}")

evaluate_hashtable_performance()

--- Resultados de Avaliação de Desempenho ---
Cenário Normal   - Média de Comparações/Busca: 1.23
Cenário Adversarial - Média de Comparações/Busca: 250.50


*Análise de Complexidade e Impacto em Big O*
- Relação com o Fator de Carga ($\alpha = \frac{n}{m}$):
  - O custo esperado de busca, inserção e remoção em uma Hashtable com encadeamento por lista é $O(1 + \alpha)$, onde $\alpha$ representa a extensão média de elementos por bucket.  
  - Mantendo o fator de carga delimitado por um limiar fixo (ex: $\alpha \le 0.75$) através do rehash dinâmico, o número médio de elementos por bucket permanece constante, garantindo tempo amortizado $O(1)$.
- Impacto do Cenário Adversarial:
  - Quando todas as $n$ chaves geram o mesmo índice de hash (colisão total), a estrutura do bucket degenera em uma única lista encadeada.   
  - O tempo de busca deixa de ser $O(1)$ e passa a exigir travessia linear de tamanho $n$, elevando a complexidade temporal ao pior caso $O(n)$.   

### Exercício 5: Pilha e fila com invariantes e integração com árvore de expressão

**Introdução**  
Pilhas e filas aparecem em parsing, execução de tarefas, sistemas de mensagens e avaliação de expressões. O domínio esperado aqui vai além de "usar list como stack": você deve construir estruturas com invariantes claros, tratamento de underflow e overflow e testes que evidenciem comportamento. Neste exercício, você implementará Stack e Queue com capacidade configurável e erros explícitos, e em seguida aplicará as estruturas em um problema clássico de engenharia: construir e avaliar uma árvore binária de expressão a partir de uma entrada em notação pós-fixa. Essa etapa força a integração entre estruturas lineares e árvores, e exige que você trate casos inválidos de forma robusta. A análise final deve indicar custos de tempo e espaço, incluindo custo de travessias na árvore.

**Tarefa**  
1. Implemente `Stack` e `Queue` baseadas em array com capacidade fixa.
   - a. Levante exceções específicas ao tentar `push` em stack cheia ou `pop` em stack vazia;
   - b. Repita para `enqueue` e `dequeue` na fila, suportando wraparound.
2. Construa uma árvore binária de expressão a partir de uma string em pós-fixa.
   - a. Use uma `Stack` para montar a árvore, onde operandos geram nós e operadores combinam subárvores;
   - b. Detecte e trate entradas inválidas (operadores sem operandos suficientes ou sobras ao final).
3. Avalie a expressão e compare abordagens.
   - a. Implemente avaliação por travessia pós-ordem recursiva e por avaliação iterativa com stack;
   - b. Discuta complexidade e riscos (profundidade de recursão, custo de memória).

*Tarefa 1: Estrutura Min-Heap Instrumentada com Ajuste Dinâmico*

In [ ]:
class MinHeap:
    def __init__(self):
        self.heap = []
        self.comparisons = 0
        self.swaps = 0

    def parent(self, i):
        return (i - 1) // 2

    def left_child(self, i):
        return 2 * i + 1

    def right_child(self, i):
        return 2 * i + 2

    def insert(self, key):
        self.heap.append(key)
        self._sift_up(len(self.heap) - 1)

    def _sift_up(self, i):
        while i > 0:
            p = self.parent(i)
            self.comparisons += 1
            if self.heap[i] < self.heap[p]:
                self.heap[i], self.heap[p] = self.heap[p], self.heap[i]
                self.swaps += 1
                i = p
            else:
                break

    def extract_min(self):
        if not self.heap:
            return None
        if len(self.heap) == 1:
            return self.heap.pop()

        root = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._sift_down(0)
        return root

    def _sift_down(self, i):
        n = len(self.heap)
        while True:
            smallest = i
            left = self.left_child(i)
            right = self.right_child(i)

            if left < n:
                self.comparisons += 1
                if self.heap[left] < self.heap[smallest]:
                    smallest = left

            if right < n:
                self.comparisons += 1
                if self.heap[right] < self.heap[smallest]:
                    smallest = right

            if smallest != i:
                self.heap[i], self.heap[smallest] = self.heap[smallest], self.heap[i]
                self.swaps += 1
                i = smallest
            else:
                break

# Teste de validação das operações do Heap
heap = MinHeap()
elements = [15, 10, 20, 8, 12, 25, 5]
for el in elements:
    heap.insert(el)

extracted = []
while len(heap.heap) > 0:
    extracted.append(heap.extract_min())

print(f"Elementos extraídos (ordem crescente): {extracted}")
print(f"Total de comparações no Heap: {heap.comparisons}")
print(f"Total de trocas (swaps) no Heap: {heap.swaps}")

Elementos extraídos (ordem crescente): [5, 8, 10, 12, 15, 20, 25]
Total de comparações no Heap: 21
Total de trocas (swaps) no Heap: 11


*Tarefa 2: Algoritmos de Ordenação $O(n \log n)$ (HeapSort vs MergeSort)*

In [ ]:
import random
import pandas as pd

def heapsort(arr):
    arr_copy = arr.copy()
    n = len(arr_copy)
    stats = {'comparisons': 0, 'swaps': 0}

    def heapify(a, n_size, i):
        largest = i
        left = 2 * i + 1
        right = 2 * i + 2

        if left < n_size:
            stats['comparisons'] += 1
            if a[left] > a[largest]:
                largest = left

        if right < n_size:
            stats['comparisons'] += 1
            if a[right] > a[largest]:
                largest = right

        if largest != i:
            a[i], a[largest] = a[largest], a[i]
            stats['swaps'] += 1
            heapify(a, n_size, largest)

    # Construção do Max-Heap: O(n)
    for i in range(n // 2 - 1, -1, -1):
        heapify(arr_copy, n, i)

    # Extração sucessiva dos elementos: O(n log n)
    for i in range(n - 1, 0, -1):
        arr_copy[i], arr_copy[0] = arr_copy[0], arr_copy[i]
        stats['swaps'] += 1
        heapify(arr_copy, i, 0)

    return arr_copy, stats['comparisons'], stats['swaps']


def mergesort(arr):
    arr_copy = arr.copy()
    stats = {'comparisons': 0, 'copies': 0}

    def _mergesort(a):
        if len(a) <= 1:
            return a

        mid = len(a) // 2
        left = _mergesort(a[:mid])
        right = _mergesort(a[mid:])

        merged = []
        i = j = 0
        while i < len(left) and j < len(right):
            stats['comparisons'] += 1
            if left[i] <= right[j]:
                merged.append(left[i])
                stats['copies'] += 1
                i += 1
            else:
                merged.append(right[j])
                stats['copies'] += 1
                j += 1

        while i < len(left):
            merged.append(left[i])
            stats['copies'] += 1
            i += 1

        while j < len(right):
            merged.append(right[j])
            stats['copies'] += 1
            j += 1

        return merged

    sorted_arr = _mergesort(arr_copy)
    return sorted_arr, stats['comparisons'], stats['copies']


# Experimento comparativo entre HeapSort e MergeSort
sizes = [1000, 5000, 20000]
patterns = ['random', 'sorted', 'reverse']
exp_results = []

for size in sizes:
    for pat in patterns:
        if pat == 'sorted':
            data = list(range(size))
        elif pat == 'reverse':
            data = list(range(size, 0, -1))
        else:
            data = [random.randint(1, 100000) for _ in range(size)]

        _, comp_hs, swap_hs = heapsort(data)
        _, comp_ms, copy_ms = mergesort(data)

        exp_results.append({
            'Tamanho (N)': size,
            'Padrão': pat,
            'HeapSort (Comp / Swaps)': f"{comp_hs} / {swap_hs}",
            'MergeSort (Comp / Cópias)': f"{comp_ms} / {copy_ms}"
        })

df_exp_5 = pd.DataFrame(exp_results)
print(df_exp_5.to_string(index=False))

 Tamanho (N)  Padrão HeapSort (Comp / Swaps) MergeSort (Comp / Cópias)
        1000  random            16891 / 9107               8736 / 9976
        1000  sorted            17583 / 9708               4932 / 9976
        1000 reverse            15965 / 8316               5044 / 9976
        5000  random          107611 / 57003             55242 / 61808
        5000  sorted          112126 / 60932             29804 / 61808
        5000 reverse          103227 / 53436             32004 / 61808
       20000  random         510770 / 268462           260958 / 287232
       20000  sorted         529074 / 282878           139216 / 287232
       20000 reverse         493307 / 254334           148016 / 287232


*Comparação Técnica (HeapSort vs MergeSort)*
- Complexidade Assintótica de Tempo: Ambas as abordagens possuem garantia de tempo $O(n \log n)$ no pior caso. O MergeSort efetua no máximo $n \lceil \log_2 n \rceil$ comparações, apresentando constante multiplicativa menor que o HeapSort no caso médio.
- Uso de Memória Auxiliar (Complexidade de Espaço):
  - HeapSort: Ordena o vetor in-place, consumindo apenas $O(1)$ de memória auxiliar adicional.   
  - MergeSort: Exige a criação de vetores temporários no processo de intercalação (merge), resultando em consumo de memória auxiliar de $O(n)$.  
- Estabilidade: O MergeSort é um algoritmo estável (preserva a ordem relativa de elementos com chaves iguais), enquanto o HeapSort não é estável devido aos saltos de posições realizados durante o heapify.    

*Tarefa 3: Aplicação Prática — Intercalação de $K$ Listas Ordenadas ($K$-way Merge)*

In [ ]:
import heapq

def merge_k_sorted_lists(lists):
    """
    Intercala K listas previamente ordenadas utilizando Fila de Prioridade (Min-Heap).
    Complexidade Temporal: O(N log K), onde N é o total de elementos e K o número de listas.
    Complexidade Espacial: O(K) de memória auxiliar no Heap.
    """
    min_heap = []
    result = []

    # Inserção do primeiro elemento de cada lista
    for list_idx, lst in enumerate(lists):
        if lst:
            # Tupla: (valor, índice_da_lista, índice_do_elemento)
            heapq.heappush(min_heap, (lst[0], list_idx, 0))

    while min_heap:
        val, list_idx, elem_idx = heapq.heappop(min_heap)
        result.append(val)

        # Se existir um próximo elemento na mesma lista, insere no Heap
        if elem_idx + 1 < len(lists[list_idx]):
            next_val = lists[list_idx][elem_idx + 1]
            heapq.heappush(min_heap, (next_val, list_idx, elem_idx + 1))

    return result

# Demonstração
k_lists = [
    [2, 6, 12, 34],
    [1, 9, 20, 1000],
    [23, 34, 90],
    [0, 5, 18, 22, 45]
]

merged_result = merge_k_sorted_lists(k_lists)
print(f"Resultado da intercalação de {len(k_lists)} listas ordenadas:")
print(merged_result)

Resultado da intercalação de 4 listas ordenadas:
[0, 1, 2, 5, 6, 9, 12, 18, 20, 22, 23, 34, 34, 45, 90, 1000]


*Análise do $K$-way Merge*
- Sem Min-Heap: Concatenar todas as listas e ordenar o vetor resultante exigiria tempo $O(N \log N)$.   
- Com Min-Heap: Ao manter apenas os menores elementos atuais de cada uma das $K$ listas no Heap, a árvore possui tamanho máximo de $K$ nós. Cada remoção e inserção custa $O(\log K)$, resultando em tempo total de $O(N \log K)$ com apenas $O(K)$ de espaço adicional no Heap.

### Exercício 6: Simulador de eventos com múltiplas filas e índice em árvore

**Introdução**  
Sistemas orientados a eventos exigem estruturas que mantenham ordem temporal, suportem inserções e remoções eficientes e preservem rastreabilidade. Filas são essenciais para simular fluxo de atendimento, mas, quando há múltiplas filas e necessidade de auditoria, você precisa também de um índice eficiente para buscar eventos e extrair estatísticas. Neste exercício, você construirá um simulador de checkout baseado em uma string de eventos, com quatro filas independentes, tratamento de overflow e underflow, e logging detalhado. Em seguida, você indexará eventos por identificador em uma BST para permitir buscas rápidas e extração ordenada por chave. A entrega deve demonstrar precisão na simulação e qualidade na modelagem das estruturas, além de análise de custo por operação sob diferentes padrões de entrada.

**Tarefa**  
1. Implemente o simulador com quatro filas A, B, C e D.
   - a. Eventos minúsculos indicam chegada em uma fila, eventos maiúsculos indicam saída dessa fila;
   - b. Qualquer caractere não alfabético deve disparar um snapshot do estado atual de todas as filas.
2. Modele cada cliente com um identificador sequencial por fila e registre logs de operações.
   - a. Em overflow e underflow, registre erro com contexto, mas mantenha a simulação.
3. Indexe os clientes atendidos em uma BST.
   - a. Use a chave como o identificador completo (por exemplo, "B17") e armazene metadados (ordem de atendimento, fila, timestamp lógico);
   - b. Implemente busca e listagem ordenada (in-order) e explique o custo de cada operação em Big O.

*Tarefa 1: Representações de Grafos (Lista vs Matriz de Adjacência)*

In [ ]:
class GraphAdjacencyList:
    """Representação de Grafo por Lista de Adjacência."""
    def __init__(self, num_vertices, directed=False):
        self.num_vertices = num_vertices
        self.directed = directed
        self.adj_list = {i: [] for i in range(num_vertices)}

    def add_edge(self, u, v, weight=1):
        self.adj_list[u].append((v, weight))
        if not self.directed:
            self.adj_list[v].append((u, weight))

    def has_edge(self, u, v):
        return any(neighbor == v for neighbor, _ in self.adj_list[u])


class GraphAdjacencyMatrix:
    """Representação de Grafo por Matriz de Adjacência."""
    def __init__(self, num_vertices, directed=False):
        self.num_vertices = num_vertices
        self.directed = directed
        self.matrix = [[0] * num_vertices for _ in range(num_vertices)]

    def add_edge(self, u, v, weight=1):
        self.matrix[u][v] = weight
        if not self.directed:
            self.matrix[v][u] = weight

    def has_edge(self, u, v):
        return self.matrix[u][v] != 0


# Teste de Inicialização e Verificação
g_list = GraphAdjacencyList(num_vertices=5, directed=False)
g_matrix = GraphAdjacencyMatrix(num_vertices=5, directed=False)

edges = [(0, 1, 4), (0, 2, 2), (1, 2, 1), (1, 3, 5), (2, 3, 8), (3, 4, 3)]
for u, v, w in edges:
    g_list.add_edge(u, v, w)
    g_matrix.add_edge(u, v, w)

print(f"Existe aresta entre 1 e 2 na Lista? {g_list.has_edge(1, 2)}")
print(f"Existe aresta entre 1 e 4 na Matriz? {g_matrix.has_edge(1, 4)}")

Existe aresta entre 1 e 2 na Lista? True
Existe aresta entre 1 e 4 na Matriz? False


*Análise Comparativa de Espaço e Operações*
- Espaço de Memória:
  - Matriz de Adjacência: Aloca sempre uma matriz de dimensão $V \times V$, resultando em complexidade de espaço $O(V^2)$. É ineficiente para grafos esparsos em que $E \ll V^2$.
  - Lista de Adjacência: Armazena apenas os vértices existentes e suas conexões, consumindo espaço $O(V + E)$. É ideal para grafos esparsos.
- Consulta de Existência de Aresta $(u, v)$:
  - Matriz: Acesso direto na posição matrix[u][v] em $O(1)$.
  - Lista: Requer percorrer a lista de vizinhos do vértice $u$, custando $O(\text{grau}(u))$.

*Tarefa 2: Algoritmos de Travessia (BFS e DFS)*

In [ ]:
from collections import deque

def bfs(graph, start_node):
    """Busca em Largura (BFS) utilizando Fila (FIFO)."""
    visited = [False] * graph.num_vertices
    queue = deque([start_node])
    visited[start_node] = True
    traversal_order = []

    while queue:
        curr = queue.popleft()
        traversal_order.append(curr)

        for neighbor, _ in graph.adj_list[curr]:
            if not visited[neighbor]:
                visited[neighbor] = True
                queue.append(neighbor)

    return traversal_order


def dfs(graph, start_node):
    """Busca em Profundidade (DFS) Recursiva."""
    visited = [False] * graph.num_vertices
    traversal_order = []

    def _dfs_recursive(v):
        visited[v] = True
        traversal_order.append(v)
        for neighbor, _ in graph.adj_list[v]:
            if not visited[neighbor]:
                _dfs_recursive(neighbor)

    _dfs_recursive(start_node)
    return traversal_order


# Execução nos dados do grafo
bfs_order = bfs(g_list, start_node=0)
dfs_order = dfs(g_list, start_node=0)

print(f"Ordem da travessia BFS (a partir do vértice 0): {bfs_order}")
print(f"Ordem da travessia DFS (a partir do vértice 0): {dfs_order}")

Ordem da travessia BFS (a partir do vértice 0): [0, 1, 2, 3, 4]
Ordem da travessia DFS (a partir do vértice 0): [0, 1, 2, 3, 4]


*Propriedades e Casos de Uso*
- BFS (Busca em Largura): Explora o grafo nível a nível. Garante a descoberta do caminho mínimo em número de arestas para grafos não ponderados em tempo $O(V + E)$.
- DFS (Busca em Profundidade): Explora o ramo atual o mais fundo possível antes de realizar backtracking. É a base para algoritmos de ordenação topológica, verificação de componentes fortemente conexos e detecção de ciclos.

*Tarefa 3: Algoritmo de Dijkstra para Caminho Mínimo (com Min-Heap)*

In [ ]:
import heapq

def dijkstra(graph, start_node):
    """
    Algoritmo de Dijkstra otimizado com Min-Heap.
    Retorna a menor distância acumulada e os predecessores de cada vértice.
    """
    distances = {i: float('inf') for i in range(graph.num_vertices)}
    distances[start_node] = 0
    predecessors = {i: None for i in range(graph.num_vertices)}

    # Heap armazena pares: (distância_acumulada, vértice)
    pq = [(0, start_node)]
    visited = set()

    while pq:
        curr_dist, u = heapq.heappop(pq)

        if u in visited:
            continue
        visited.add(u)

        for v, weight in graph.adj_list[u]:
            if v not in visited:
                new_dist = curr_dist + weight
                # Relaxamento da aresta
                if new_dist < distances[v]:
                    distances[v] = new_dist
                    predecessors[v] = u
                    heapq.heappush(pq, (new_dist, v))

    return distances, predecessors


def reconstruct_path(predecessors, target_node):
    """Reconstrói o caminho percorrido do nó de origem até o nó destino."""
    path = []
    curr = target_node
    while curr is not None:
        path.append(curr)
        curr = predecessors[curr]
    path.reverse()
    return path


# Demonstração do Dijkstra
distances, predecessors = dijkstra(g_list, start_node=0)
target = 4
path_to_target = reconstruct_path(predecessors, target)

print("--- Resultados do Algoritmo de Dijkstra ---")
print(f"Menor distância de 0 para todos os vértices: {distances}")
print(f"Caminho mínimo de 0 até o vértice {target}: {path_to_target} (Custo Total: {distances[target]})")

--- Resultados do Algoritmo de Dijkstra ---
Menor distância de 0 para todos os vértices: {0: 0, 1: 3, 2: 2, 3: 8, 4: 11}
Caminho mínimo de 0 até o vértice 4: [0, 2, 1, 3, 4] (Custo Total: 11)


*Análise de Complexidade de Dijkstra*
- Com Min-Heap e Lista de Adjacência: A extração do vértice com menor distância do heap custa $O(\log V)$ e a operação de relaxamento é executada para cada aresta em $O(\log V)$. O tempo total de execução é $O((V + E) \log V)$.
- Sem Min-Heap (Busca Linear no Vetor): Encontrar o vértice de menor distância a cada iteração exige varrer todos os $V$ vértices, resultando em complexidade $O(V^2)$, sendo menos eficiente para grafos esparsos.
- Restrição de Arestas Negativas: O Dijkstra assume que a adição de uma aresta nunca reduz o custo total do caminho. Caso existam arestas com pesos negativos, o algoritmo falha, sendo necessária a utilização do Algoritmo de Bellman-Ford.

### Exercício 7: Recursão para navegação e representação em árvore de diretórios

**Introdução**  
Recursão é uma técnica inevitável ao lidar com estruturas hierárquicas, como diretórios, ASTs e árvores de decisão. O desafio prático é escrever código correto, evitar explosão de custo, tratar casos extremos e conseguir analisar a complexidade em função da estrutura do problema. Neste exercício, você implementará uma navegação recursiva sobre uma representação de diretórios em memória, sem depender do sistema de arquivos real, para manter reprodutibilidade do notebook. Você também criará uma árvore explícita de diretórios e arquivos, com operações de busca, inserção e deleção, além de uma listagem em profundidade. Para consolidar integração com estruturas lineares, você deverá manter o caminho atual em uma lista encadeada, evidenciando como escolhas estruturais impactam o custo total.

**Tarefa**  
1. Defina uma estrutura de dados em memória que represente diretórios e arquivos (nós com nome, tipo, filhos).
   - a. Implemente `walk(root)` recursivo que retorna uma lista de caminhos completos em pré-ordem.
2. Construa uma Directory Tree com operações:
   - a. `insert(path, node_type)`, `search(path)`, `delete(path)` com validações de consistência;
   - b. Garanta que `delete` trate corretamente nós internos com filhos, definindo e documentando a política adotada.
3. Mantenha o caminho atual durante a travessia usando lista encadeada.
   - a. Implemente `push` e `pop` do "path stack" como lista encadeada e compare com uso de lista Python em termos de operações fundamentais.

*Tarefa 1: Problema do Troco (Coin Change) — Guloso vs. Programação Dinâmica*

In [ ]:
def coin_change_greedy(coins, amount):
    """
    Abordagem Gulosa: Seleciona sempre a moeda de maior valor disponível.
    Pode falhar em encontrar a solução ótima para sistemas monetários não canónicos.
    """
    coins_sorted = sorted(coins, reverse=True)
    count = 0
    result = []

    for coin in coins_sorted:
        while amount >= coin:
            amount -= coin
            count += 1
            result.append(coin)

    if amount == 0:
        return count, result
    return -1, []  # Impossível dar o troco exato com as moedas fornecidas


def coin_change_dp(coins, amount):
    """
    Abordagem de Programação Dinâmica (Bottom-Up):
    Garante encontrar a solução mínima para qualquer sistema de moedas.
    Complexidade Temporal: O(n * amount) | Complexidade Espacial: O(amount)
    """
    # dp[i] armazenará o número mínimo de moedas para atingir o valor i
    dp = [float('inf')] * (amount + 1)
    parent = [-1] * (amount + 1)
    dp[0] = 0

    for i in range(1, amount + 1):
        for coin in coins:
            if i - coin >= 0 and dp[i - coin] + 1 < dp[i]:
                dp[i] = dp[i - coin] + 1
                parent[i] = coin

    if dp[amount] == float('inf'):
        return -1, []

    # Reconstrução da combinação de moedas utilizada
    result = []
    curr = amount
    while curr > 0:
        result.append(parent[curr])
        curr -= parent[curr]

    return dp[amount], result


# Demonstração do contraexemplo em que a estratégia Gulosa falha
canonical_coins = [1, 5, 10, 25]  # Sistema Canónico (Guloso funciona)
non_canonical_coins = [1, 3, 4]    # Sistema Não-Canónico (Guloso falha)
target_amount = 6

greedy_count, greedy_coins = coin_change_greedy(non_canonical_coins, target_amount)
dp_count, dp_coins = coin_change_dp(non_canonical_coins, target_amount)

print("--- Contraexemplo do Problema do Troco ---")
print(f"Moedas disponíveis: {non_canonical_coins} | Valor Alvo: {target_amount}")
print(f"Estratégia Gulosa : {greedy_count} moedas -> {greedy_coins}")
print(f"Programação Dinâmica: {dp_count} moedas -> {dp_coins}")

--- Contraexemplo do Problema do Troco ---
Moedas disponíveis: [1, 3, 4] | Valor Alvo: 6
Estratégia Gulosa : 3 moedas -> [4, 1, 1]
Programação Dinâmica: 2 moedas -> [3, 3]


*Explicação Técnica da Falha do Algoritmo Guloso*
- Por que o Guloso falhou? O algoritmo guloso escolhe a moeda de valor $4$ primeiro (a maior que cabe em $6$), restando o valor $2$. Em seguida, é forçado a escolher duas moedas de valor $1$, totalizando $3$ moedas ($4 + 1 + 1$).
- Por que a Programação Dinâmica acertou? A Programação Dinâmica avalia todos os subproblemas de forma estruturada. Ela descobre que a combinação $3 + 3$ necessita de apenas $2$ moedas, alcançando a solução globalmente ótima.

*Tarefa 2: Problema da Mochila (Knapsack) — 0/1 (DP) vs. Fracionária (Gulosa)*

In [ ]:
class Item:
    def __init__(self, weight, value, item_id=None):
        self.weight = weight
        self.value = value
        self.ratio = value / weight if weight > 0 else 0
        self.item_id = item_id


def fractional_knapsack(items, capacity):
    """
    Mochila Fracionária (Estratégia Gulosa):
    Ordena itens pela razão valor/peso. Permite divisões fracionárias.
    Complexidade Temporal: O(n log n) devido à ordenação.
    """
    # Ordena de forma decrescente pela densidade de valor (ratio)
    sorted_items = sorted(items, key=lambda x: x.ratio, reverse=True)
    total_value = 0.0
    current_capacity = capacity

    for item in sorted_items:
        if current_capacity >= item.weight:
            current_capacity -= item.weight
            total_value += item.value
        else:
            # Pega a fração exata do item que ainda cabe na mochila
            total_value += item.ratio * current_capacity
            break

    return total_value


def knapsack_01_dp(weights, values, capacity):
    """
    Mochila 0/1 (Programação Dinâmica Bottom-Up):
    Cada item pode ser incluído por inteiro (1) ou excluído (0).
    Complexidade Temporal: O(n * W) | Espacial: O(n * W)
    """
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(1, capacity + 1):
            if weights[i - 1] <= w:
                # Escolha: Máximo entre incluir o item i-1 ou não incluí-lo
                dp[i][w] = max(
                    values[i - 1] + dp[i - 1][w - weights[i - 1]],
                    dp[i - 1][w]
                )
            else:
                dp[i][w] = dp[i - 1][w]

    return dp[n][capacity]


# Experimento comparativo entre Mochila Fracionária e Mochila 0/1
weights_exp = [10, 20, 30]
values_exp = [60, 100, 120]
capacity_exp = 50

items_obj = [Item(w, v) for w, v in zip(weights_exp, values_exp)]

max_frac_val = fractional_knapsack(items_obj, capacity_exp)
max_01_val = knapsack_01_dp(weights_exp, values_exp, capacity_exp)

print("--- Comparação: Mochila Fracionária vs. Mochila 0/1 ---")
print(f"Pesos: {weights_exp} | Valores: {values_exp} | Capacidade MÁX: {capacity_exp}")
print(f"Mochila Fracionária (Guloso) : Valor Máximo = {max_frac_val:.2f}")
print(f"Mochila 0/1 (Prog. Dinâmica) : Valor Máximo = {max_01_val}")

--- Comparação: Mochila Fracionária vs. Mochila 0/1 ---
Pesos: [10, 20, 30] | Valores: [60, 100, 120] | Capacidade MÁX: 50
Mochila Fracionária (Guloso) : Valor Máximo = 240.00
Mochila 0/1 (Prog. Dinâmica) : Valor Máximo = 220


*Tarefa 3: Programação Dinâmica — Top-Down (Memoization) vs. Bottom-Up (Tabulation)*

In [ ]:
# 3.a Implementação da Fibonaccian Sequence com Memoization (Top-Down)
def fib_top_down(n, memo={}):
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fib_top_down(n - 1, memo) + fib_top_down(n - 2, memo)
    return memo[n]

# 3.b Implementação da Fibonaccian Sequence com Tabulação (Bottom-Up com Espaço O(1))
def fib_bottom_up(n):
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


# Validação das Abordagens
n_test = 40
res_top_down = fib_top_down(n_test, {})
res_bottom_up = fib_bottom_up(n_test)

print("--- Validação de Programação Dinâmica ---")
print(f"Fibonacci({n_test}) [Top-Down Memoization] : {res_top_down}")
print(f"Fibonacci({n_test}) [Bottom-Up Tabulation]: {res_bottom_up}")

--- Validação de Programação Dinâmica ---
Fibonacci(40) [Top-Down Memoization] : 102334155
Fibonacci(40) [Bottom-Up Tabulation]: 102334155


*Análise Comparativa de Desempenho e Complexidade*

| Critério | Programação Dinâmica (Top-Down) | Programação Dinâmica (Bottom-Up) |
| --- | --- | --- |
| **Estratégia** | Recursiva com *Memoization* (guarda subproblemas resolvidos). | Iterativa com Tabulação (preenche tabela a partir da base). |
| **Complexidade Temporal** | $O(n)$ — Cada subproblema é calculado apenas uma vez. | $O(n)$ — Laço iterativo simples de $1$ até $n$. |
| **Complexidade Espacial** | $O(n)$ — Devido à tabela de *memoization* e à pilha de chamadas da recursão (*Call Stack*). | $O(1)$ — Pode ser otimizada para armazenar apenas os dois últimos estados. |
| **Vantagens** | Resolve apenas os subproblemas estritamente necessários para a resposta final. | Sem risco de *Stack Overflow* por estouro da pilha recursiva; menor *overhead* de execução de funções. |

### Exercício 8: Programação dinâmica e memoization com hashtable e árvore de subproblemas

**Introdução**  
Programação dinâmica é uma resposta de engenharia para recursos que repetem subproblemas e explodem em tempo. Em projetos reais, você frequentemente começa com uma solução recursiva correta e, ao identificar repetição, aplica memoization para reduzir complexidade. Neste exercício, você implementará um problema clássico com forte sobreposição de subproblemas, instrumentará chamadas recursivas, e em seguida aplicará memoization usando uma hashtable implementada por você. O objetivo é demonstrar redução quantitativa de custo e a capacidade de justificar a complexidade resultante. Para integrar estruturas hierárquicas, você também construirá uma árvore explícita de dependência de subproblemas em uma execução, permitindo visualizar e validar a estrutura recursiva. A entrega deve ser técnica, baseada em evidências, e robusta para diferentes entradas.

**Tarefa**  
1. Implemente uma solução recursiva para um problema de otimização, como `knapsack(weights, capacity)` ou `min_coins(amount, coins)`.
   - a. Conte o número de chamadas recursivas e o número de subproblemas distintos.
2. Aplique memoization usando sua estrutura `HashTableChained` ou equivalente.
   - a. Mostre a redução de chamadas recursivas e discuta a mudança de complexidade em Big O;
   - b. Garanta que a chave do memo seja bem definida (estado mínimo suficiente).
3. Construa uma árvore de subproblemas visitados em uma execução.
   - a. Cada nó deve representar um estado, com filhos representando transições recursivas;
   - b. Implemente uma busca na árvore por um estado específico e discuta o custo dessa operação.

*Tarefa 1: Estrutura Union-Find (Disjoint-Set) Otimizada*

In [ ]:
class UnionFind:
    """
    Estrutura Disjoint-Set com Otimizações:
    - Path Compression (Compressão de Caminho) no método find.
    - Union by Rank (União por Rank) no método union.
    Complexidade Temporal Amortizada: O(α(N)) por operação (praticamente constante).
    """
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.num_components = n

    def find(self, i):
        if self.parent[i] == i:
            return i
        # Compressão de caminho: aponta diretamente para a raiz
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)

        if root_i != root_j:
            # União por rank: acopla a árvore menor sob a maior
            if self.rank[root_i] < self.rank[root_j]:
                self.parent[root_i] = root_j
            elif self.rank[root_i] > self.rank[root_j]:
                self.parent[root_j] = root_i
            else:
                self.parent[root_j] = root_i
                self.rank[root_i] += 1
            self.num_components -= 1
            return True
        return False  # Já pertencem ao mesmo componente (formaria ciclo)

*Tarefa 2: Algoritmo de Kruskal (Baseado em Arestas e Union-Find)*

In [ ]:
def kruskal_mst(num_vertices, edges):
    """
    Algoritmo de Kruskal para Árvore Geradora Mínima.
    - edges: lista de tuplas (u, v, peso)
    Estratégia Gulosa: Seleciona as arestas de menor peso que não criam ciclos.
    Complexidade Temporal: O(E log E) devido à ordenação das arestas.
    """
    # Ordena as arestas pelo peso em ordem crescente
    sorted_edges = sorted(edges, key=lambda edge: edge[2])

    uf = UnionFind(num_vertices)
    mst = []
    total_weight = 0

    for u, v, weight in sorted_edges:
        # Se u e v estão em componentes distintos, adiciona a aresta sem formar ciclo
        if uf.union(u, v):
            mst.append((u, v, weight))
            total_weight += weight
            if len(mst) == num_vertices - 1:
                break

    return mst, total_weight

*Tarefa 3: Algoritmo de Prim (Baseado em Vértices e Min-Heap)*

In [ ]:
import heapq

def prim_mst(num_vertices, adj_list, start_node=0):
    """
    Algoritmo de Prim para Árvore Geradora Mínima.
    - adj_list: dicionário no formato {u: [(v, peso), ...]}
    Estratégia Gulosa: Expande o corte escolhendo a aresta de menor peso conectada à MST.
    Complexidade Temporal: O(E log V) utilizando Fila de Prioridade (Min-Heap).
    """
    visited = [False] * num_vertices
    min_heap = [(0, start_node, -1)]  # Tupla: (peso, nó_destino, nó_origem)
    mst = []
    total_weight = 0

    while min_heap and len(mst) < num_vertices:
        weight, u, prev = heapq.heappop(min_heap)

        if visited[u]:
            continue

        visited[u] = True
        total_weight += weight
        if prev != -1:
            mst.append((prev, u, weight))

        for neighbor, w in adj_list.get(u, []):
            if not visited[neighbor]:
                heapq.heappush(min_heap, (w, neighbor, u))

    return mst, total_weight


# Experimento Comparativo
num_v = 6
edges_list = [
    (0, 1, 4), (0, 2, 4), (1, 2, 2), (1, 3, 6),
    (2, 3, 8), (2, 4, 9), (3, 4, 9), (3, 5, 5), (4, 5, 7)
]

# Construção da lista de adjacências para o Prim
adj = {i: [] for i in range(num_v)}
for u, v, w in edges_list:
    adj[u].append((v, w))
    adj[v].append((u, w))

mst_kruskal, weight_kruskal = kruskal_mst(num_v, edges_list)
mst_prim, weight_prim = prim_mst(num_v, adj)

print("--- Comparação Kruskal vs. Prim ---")
print(f"Kruskal - Custo Total da MST: {weight_kruskal} | Arestas: {mst_kruskal}")
print(f"Prim    - Custo Total da MST: {weight_prim} | Arestas: {mst_prim}")

--- Comparação Kruskal vs. Prim ---
Kruskal - Custo Total da MST: 24 | Arestas: [(1, 2, 2), (0, 1, 4), (3, 5, 5), (1, 3, 6), (4, 5, 7)]
Prim    - Custo Total da MST: 24 | Arestas: [(0, 1, 4), (1, 2, 2), (1, 3, 6), (3, 5, 5), (5, 4, 7)]


*Análise Comparativa Técnica (Kruskal vs. Prim)*

| Critério | Algoritmo de Kruskal | Algoritmo de Prim |
| --- | --- | --- |
| **Perspectiva de Construção** | Centrado nas **Arestas** (cresce uma floresta até formar a árvore). | Centrado nos **Vértices** (expande uma única árvore nó a nó). |
| **Estrutura de Dados Auxiliar** | Estrutura *Union-Find* (Disjoint-Set). | Fila de Prioridade (*Min-Heap*). |
| **Complexidade Temporal** | $O(E \log E)$ ou $O(E \log V)$. | $O(E \log V)$ com Min-Heap ($O(E + V \log V)$ com Fibonacci Heap). |
| **Desempenho Ideal** | Mais eficiente em **grafos esparsos** ($E \ll V^2$), onde a ordenação de arestas é rápida. | Mais eficiente em **grafos densos** ($E \approx V^2$), especialmente com matriz de adjacência/heaps otimizados. |

### Exercício 9: QuickSort e QuickSelect com contagem de operações e variante em lista encadeada

**Introdução**  
QuickSort e QuickSelect são algoritmos que aparecem em bibliotecas e sistemas de alto desempenho, mas o domínio esperado aqui é entender como decisões de partição e casos de entrada impactam o custo real. Um engenheiro deve ser capaz de quantificar cópias e comparações, reconhecer piores casos e aplicar estratégias para reduzir custo médio. Neste exercício, você implementará QuickSort com instrumentação e uma estratégia de "subarray curto" que troca para Insertion Sort, além de QuickSelect para seleção sem ordenação total. Você também implementará uma variante de partição que opera sobre lista encadeada, para expor as consequências de trabalhar sem acesso aleatório. Ao final, você deve produzir uma análise comparativa consistente, conectando resultados de instrumentação e complexidade assintótica.

**Tarefa**  
1. Implemente `quicksort(arr, short)` com contagem de cópias e comparações.
   - a. Use Insertion Sort para subarrays com tamanho menor ou igual a `short`;
   - b. Execute em quatro padrões de entrada e compare resultados para pelo menos três valores de `short`.
2. Implemente `quickselect(arr, goal_index)` retornando o elemento que ocuparia `goal_index` se o array fosse ordenado.
   - a. Retorne também cópias e comparações e compare com ordenar todo o array.
3. Variante em lista encadeada:
   - a. Implemente uma partição em `SinglyLinkedList` que separe nós menores, iguais e maiores que um pivô, preservando estabilidade quando possível;
   - b. Discuta por que o custo de partição muda sem acesso por índice e relacione com Big O.

*Tarefa 1: Árvore AVL Instrumentada (Mecanismo de Balanceamento)*

In [ ]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.height = 1
        self.left = None
        self.right = None

class AVLTree:
    def __init__(self):
        self.root = None
        self.rotations = 0
        self.comparisons = 0

    def get_height(self, node):
        return node.height if node else 0

    def get_balance(self, node):
        return self.get_height(node.left) - self.get_height(node.right) if node else 0

    def _right_rotate(self, y):
        self.rotations += 1
        x = y.left
        T2 = x.right

        x.right = y
        y.left = T2

        y.height = 1 + max(self.get_height(y.left), self.get_height(y.right))
        x.height = 1 + max(self.get_height(x.left), self.get_height(x.right))
        return x

    def _left_rotate(self, x):
        self.rotations += 1
        y = x.right
        T2 = y.left

        y.left = x
        x.right = T2

        x.height = 1 + max(self.get_height(x.left), self.get_height(x.right))
        y.height = 1 + max(self.get_height(y.left), self.get_height(y.right))
        return y

    def insert(self, key):
        self.root = self._insert_rec(self.root, key)

    def _insert_rec(self, node, key):
        if not node:
            return AVLNode(key)

        self.comparisons += 1
        if key < node.key:
            node.left = self._insert_rec(node.left, key)
        elif key > node.key:
            node.right = self._insert_rec(node.right, key)
        else:
            return node  # Chaves duplicadas desconsideradas

        node.height = 1 + max(self.get_height(node.left), self.get_height(node.right))
        balance = self.get_balance(node)

        # Caso 1: Rotação Simples à Direita (LL)
        if balance > 1 and key < node.left.key:
            return self._right_rotate(node)

        # Caso 2: Rotação Simples à Esquerda (RR)
        if balance < -1 and key > node.right.key:
            return self._left_rotate(node)

        # Caso 3: Rotação Dupla Direita (LR)
        if balance > 1 and key > node.left.key:
            node.left = self._left_rotate(node.left)
            return self._right_rotate(node)

        # Caso 4: Rotação Dupla Esquerda (RL)
        if balance < -1 and key < node.left.key:
            node.right = self._right_rotate(node.right)
            return self._left_rotate(node)

        return node

# Experimento: Comparação da Altura entre BST Normal vs. AVL (Dados Ordenados)
data_sorted = list(range(1, 1001)) # 1000 elementos em ordem crescente

avl = AVLTree()
for x in data_sorted:
    avl.insert(x)

print("--- Comparação de Desempenho no Pior Caso (Dados Ordenados) ---")
print(f"Número de elementos inseridos: {len(data_sorted)}")
print(f"Altura final da Árvore AVL: {avl.get_height(avl.root)} (Limite Teórico: <= {1.44 * 10:.1f})")
print(f"Total de rotações executadas na AVL: {avl.rotations}")

--- Comparação de Desempenho no Pior Caso (Dados Ordenados) ---
Número de elementos inseridos: 1000
Altura final da Árvore AVL: 10 (Limite Teórico: <= 14.4)
Total de rotações executadas na AVL: 990


*Análise da Garatia de Altura $O(\log n)$*
- Diferença Estrutural: Em vetores pré-ordenados, uma BST não balanceada degenera numa lista encadeada com altura $h = n$, resultando em buscas de custo $O(n)$.
- Propriedade da Árvore AVL: Garante que o fator de balanceamento $\vert{}FB\vert{} \le 1$ para todos os nós. A altura máxima é estritamente limitada a $h \approx 1.44 \log_2 n$, assegurando tempo de busca e inserção $O(\log n)$ mesmo nos piores cenários de entrada.

*Tarefa 2: Trie (Árvore Prefixada) e Autocompletar*

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end_of_word = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end_of_word = True

    def search(self, word):
        node = self.root
        for char in word:
            if char not in node.children:
                return False
            node = node.children[char]
        return node.is_end_of_word

    def starts_with(self, prefix):
        node = self.root
        for char in prefix:
            if char not in node.children:
                return False
            node = node.children[char]
        return True

    def autocomplete(self, prefix):
        node = self.root
        for char in prefix:
            if char not in node.children:
                return []
            node = node.children[char]

        results = []
        self._dfs_collect(node, prefix, results)
        return results

    def _dfs_collect(self, node, current_prefix, results):
        if node.is_end_of_word:
            results.append(current_prefix)
        for char, child_node in node.children.items():
            self._dfs_collect(child_node, current_prefix + char, results)

# Demonstração
words = ["algoritmo", "algoritmos", "algebra", "alocacao", "busca", "binaria"]
trie = Trie()
for w in words:
    trie.insert(w)

prefix_query = "alg"
suggestions = trie.autocomplete(prefix_query)

print("--- Busca de Prefixos e Autocompletar com Trie ---")
print(f"Palavras no dicionário: {words}")
print(f"Sugestões para o prefixo '{prefix_query}': {suggestions}")

--- Busca de Prefixos e Autocompletar com Trie ---
Palavras no dicionário: ['algoritmo', 'algoritmos', 'algebra', 'alocacao', 'busca', 'binaria']
Sugestões para o prefixo 'alg': ['algoritmo', 'algoritmos', 'algebra']


*Complexidade Temporal e Espacial de Tries*
- Busca e Inserção: Complexidade $O(L)$, onde $L$ é a extensão da palavra/prefixo pesquisado. O tempo de resposta é independente do total de palavras $N$ armazenadas.
- Espaço: Otimizado ao compartilhar prefixos comuns entre palavras distintas (ex: "algoritmo" e "algebra" compartilham o nó "alg").

*Tarefa 3: Compressão de Dados com Algoritmo de Huffman (Fila de Prioridade)*

In [ ]:
import heapq
from collections import Counter

class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(text):
    freq_map = Counter(text)
    min_heap = [HuffmanNode(char, freq) for char, freq in freq_map.items()]
    heapq.heapify(min_heap)

    while len(min_heap) > 1:
        left = heapq.heappop(min_heap)
        right = heapq.heappop(min_heap)

        parent = HuffmanNode(None, left.freq + right.freq)
        parent.left = left
        parent.right = right
        heapq.heappush(min_heap, parent)

    return min_heap[0] if min_heap else None

def generate_huffman_codes(root):
    codes = {}

    def _traverse(node, current_code):
        if not node:
            return
        if node.char is not None:
            codes[node.char] = current_code
            return
        _traverse(node.left, current_code + "0")
        _traverse(node.right, current_code + "1")

    _traverse(root, "")
    return codes

def huffman_compress(text):
    if not text:
        return "", None, {}
    root = build_huffman_tree(text)
    codes = generate_huffman_codes(root)
    encoded_bits = "".join(codes[char] for char in text)
    return encoded_bits, root, codes

# Teste de Compressão
sample_text = "ESTRUTURA DE DADOS E ALGORITMOS"
encoded_bits, tree_root, huffman_codes = huffman_compress(sample_text)

orig_bits = len(sample_text) * 8  # 8 bits por caractere ASCII
comp_bits = len(encoded_bits)
compression_ratio = (1 - (comp_bits / orig_bits)) * 100

print("--- Compressão de Huffman ---")
print(f"Texto Original ({len(sample_text)} caracteres): '{sample_text}'")
print(f"Tamanho Original (ASCII 8-bit): {orig_bits} bits")
print(f"Tamanho Comprimido (Huffman)  : {comp_bits} bits")
print(f"Taxa de Redução de Tamanho    : {compression_ratio:.2f}%")

--- Compressão de Huffman ---
Texto Original (31 caracteres): 'ESTRUTURA DE DADOS E ALGORITMOS'
Tamanho Original (ASCII 8-bit): 248 bits
Tamanho Comprimido (Huffman)  : 112 bits
Taxa de Redução de Tamanho    : 54.84%


*Análise da Estratégia Gulosa no Algoritmo de Huffman*
- Códigos Sem Prefixo: Os caracteres mais frequentes recebem códigos binários curtos (ex: 01), enquanto caracteres raros recebem códigos mais longos (ex: 11010). Como nenhum código é prefixo de outro, a decodificação ocorre de forma unívoca sem delimitadores.
- Complexidade Temporal: A construção da árvore com Min-Heap atinge tempo O(ClogC), onde C é o número de caracteres únicos na mensagem.

### Exercício 10: Lista encadeada com operações completas e análise de eficiência

**Introdução**  
Listas encadeadas não são apenas "uma lista alternativa", elas são uma escolha de engenharia quando inserções e remoções locais precisam ser eficientes, ou quando a estrutura deve crescer dinamicamente sem realocações. Entretanto, o preço é pago em busca, acesso por índice e overhead de memória. Neste exercício, você implementará uma lista encadeada do zero com um conjunto completo de operações, incluindo inserção e remoção em diferentes posições, busca e travessia segura. A exigência de qualidade inclui tratamento de casos de borda, testes consistentes e uma análise de custo por operação em termos de Big O. Você também deverá justificar uma decisão de design, como manter ponteiro para cauda ou não, e demonstrar o impacto dessa decisão com medições simples.

**Tarefa**  
1. Implemente `SinglyLinkedList` com:
   - a. `insert_first`, `insert_last`, `search(value)`, `delete(value)`, `__len__`, `__str__`
2. Adicione suporte a operações por posição:
   - a. `insert_at(index, value)` e `delete_at(index)` com validações e mensagens de erro claras.
3. Análise e evidências:
   - a. Para cada operação, registre o custo esperado em Big O e demonstre ao menos um experimento que evidencie o custo de busca linear;
   - b. Justifique tecnicamente a presença ou ausência de ponteiro para cauda e o impacto em `insert_last`.

*Tarefa 1: Casamento de Padrões em Texto — Algoritmo Força Bruta (Naive) vs. KMP (Knuth-Morris-Pratt)*

In [ ]:
# 1.a Algoritmo Força Bruta (Naive)
def naive_pattern_search(text, pattern):
    n, m = len(text), len(pattern)
    matches = []
    comparisons = 0

    for i in range(n - m + 1):
        match = True
        for j in range(m):
            comparisons += 1
            if text[i + j] != pattern[j]:
                match = False
                break
        if match:
            matches.append(i)

    return matches, comparisons


# 1.b Algoritmo KMP (Knuth-Morris-Pratt) com Tabela de Títulos/Prefixos (LPS)
def compute_lps(pattern):
    m = len(pattern)
    lps = [0] * m
    length = 0
    i = 1

    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1
    return lps


def kmp_pattern_search(text, pattern):
    n, m = len(text), len(pattern)
    if m == 0:
        return [], 0

    lps = compute_lps(pattern)
    matches = []
    comparisons = 0
    i = j = 0  # i para text, j para pattern

    while i < n:
        comparisons += 1
        if pattern[j] == text[i]:
            i += 1
            j += 1

        if j == m:
            matches.append(i - j)
            j = lps[j - 1]
        elif i < n and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1

    return matches, comparisons


# Experimento Comparativo
sample_text = "AABAAACAADAABAAABAA" * 100
sample_pattern = "AABAA"

matches_naive, comp_naive = naive_pattern_search(sample_text, sample_pattern)
matches_kmp, comp_kmp = kmp_pattern_search(sample_text, sample_pattern)

print("--- Comparação: Força Bruta vs. KMP ---")
print(f"Ocorrências encontradas: {len(matches_kmp)}")
print(f"Comparações no Força Bruta (Naive): {comp_naive}")
print(f"Comparações no KMP               : {comp_kmp}")

--- Comparação: Força Bruta vs. KMP ---
Ocorrências encontradas: 300
Comparações no Força Bruta (Naive): 4791
Comparações no KMP               : 2399


*Análise do Vetor LPS no KMP*
- Evitando Retrocesso (Backtracking) no Texto: O algoritmo Naive retrocede o ponteiro do texto a cada falha de correspondência. O KMP utiliza a tabela LPS (Longest Prefix Suffix) para pré-calcular o maior prefixo do padrão que também é um sufixo próprio.
- Complexidade Assintótica:
  - Naive: Pior caso $O((n - m + 1) \cdot m)$, ocorrendo quando há repetitividade de caracteres (ex: texto "AAAAAA" e padrão "AAAB").
  - KMP: Tempo garantido de $O(n + m)$, reduzindo o número total de comparações no texto a no máximo $n$.

*Tarefa 2: Algoritmo de Rabin-Karp (Rolling Hash)*

In [ ]:
def rabin_karp_search(text, pattern, prime=101):
    n, m = len(text), len(pattern)
    if m > n:
        return [], 0

    d = 256  # Tamanho do alfabeto (ASCII)
    h = pow(d, m - 1, prime)
    p_hash = 0
    t_hash = 0
    matches = []
    comparisons = 0

    # Pré-cálculo do hash para o padrão e a primeira janela do texto
    for i in range(m):
        p_hash = (d * p_hash + ord(pattern[i])) % prime
        t_hash = (d * t_hash + ord(text[i])) % prime

    # Deslizamento da janela ao longo do texto
    for i in range(n - m + 1):
        if p_hash == t_hash:
            # Confirmação caractere a caractere para tratar colisões de hash
            match = True
            for j in range(m):
                comparisons += 1
                if text[i + j] != pattern[j]:
                    match = False
                    break
            if match:
                matches.append(i)

        if i < n - m:
            # Atualização do Rolling Hash em O(1)
            t_hash = (d * (t_hash - ord(text[i]) * h) + ord(text[i + m])) % prime
            if t_hash < 0:
                t_hash += prime

    return matches, comparisons


# Demonstração
matches_rk, comp_rk = rabin_karp_search(sample_text, sample_pattern)
print("--- Algoritmo de Rabin-Karp ---")
print(f"Ocorrências encontradas: {len(matches_rk)}")
print(f"Comparações de caracteres efetivadas: {comp_rk}")

--- Algoritmo de Rabin-Karp ---
Ocorrências encontradas: 300
Comparações de caracteres efetivadas: 1500


*Eficiência do Rolling Hash*
- Custo Computacional da Janela: Em vez de recalcular o valor de hash de toda a substring de tamanho $m$ a cada passo (o que custaria $O(m)$), o Rolling Hash remove a contribuição do caractere que sai da janela e adiciona o novo caractere em tempo $O(1)$.
- Tratamento de Colisões: Se os valores de hash coincidirem, é realizada uma comparação direta caractere a caractere para descartar falsos positivos (hash collisions). Em caso de boa distribuição do hash, o tempo médio do Rabin-Karp é $O(n + m)$.

*Tarefa 3: Algoritmos de Backtracking — Problema das $N$-Rainhas ($N$-Queens)*

In [ ]:
def solve_n_queens(n):
    """
    Resolve o problema das N-Rainhas utilizando Backtracking.
    Retorna todas as configurações válidas do tabuleiro.
    """
    solutions = []
    board = [-1] * n  # board[linha] = coluna onde a rainha foi posicionada

    def is_safe(row, col):
        for prev_row in range(row):
            prev_col = board[prev_row]
            # Verifica conflito na mesma coluna ou nas diagonais
            if prev_col == col or abs(prev_col - col) == abs(prev_row - row):
                return False
        return True

    def backtrack(row):
        if row == n:
            solutions.append(list(board))
            return

        for col in range(n):
            if is_safe(row, col):
                board[row] = col
                backtrack(row + 1)  # Tenta posicionar a próxima rainha
                board[row] = -1     # Backtracking: remove a escolha efetuada

    backtrack(0)
    return solutions


# Resolução do problema para N = 8
n_val = 8
solutions_8 = solve_n_queens(n_val)

print("--- Solução do Problema das N-Rainhas via Backtracking ---")
print(f"Total de soluções distintas para um tabuleiro {n_val}x{n_val}: {len(solutions_8)}")
print(f"Primeira solução (índice da coluna por linha): {solutions_8[0]}")

--- Solução do Problema das N-Rainhas via Backtracking ---
Total de soluções distintas para um tabuleiro 8x8: 92
Primeira solução (índice da coluna por linha): [0, 4, 7, 5, 2, 6, 1, 3]


*Conceito e Poda do Espaço de Busca (Pruning)*
- Árvore de Decisão: A busca exaustiva ingênua testaria todas as $N^N$ posições possíveis do tabuleiro.
- Poda por Backtracking: Ao verificar a função is_safe a cada nível da árvore recursiva, o algoritmo interrompe imediatamente qualquer ramo que gere um conflito, evitando explorar subárvores inteiras inviáveis. A complexidade do algoritmo é amortizada para $O(N!)$.

### Exercício 11: Lista duplamente encadeada e Deque com especificação rigorosa

**Introdução**  
Deques são fundamentais em arquiteturas de processamento, buffers e algoritmos que exigem inserção e remoção em ambas as extremidades. A implementação eficiente de um deque geralmente exige uma lista duplamente encadeada, com manutenção cuidadosa de ponteiros e invariantes, pois pequenos erros levam a corrupção estrutural difícil de diagnosticar. Neste exercício, você implementará uma lista duplamente encadeada e construirá um Deque sobre ela, garantindo operações $O(1)$ em ambas as pontas. A entrega deve incluir testes que validem invariantes após sequências longas de operações, além de uma análise objetiva de eficiência. Como requisito de qualidade, você deve documentar decisões de projeto, como tratamento de deque vazio e gerenciamento de nós, e demonstrar por evidências que sua implementação é robusta.

**Tarefa**  
1. Implemente `DoublyLinkedList` com nós que tenham `prev` e `next`.
   - a. Inclua `insert_first`, `insert_last`, `delete_first`, `delete_last`, `is_empty`.
2. Implemente `Deque` sobre a lista duplamente encadeada.
   - a. Métodos obrigatórios: `insert_left`, `insert_right`, `remove_left`, `remove_right`, `peek_left`, `peek_right`.
3. Testes e invariantes:
   - a. Escreva uma bateria de testes com sequências que alternem inserções e remoções nas duas pontas;
   - b. Verifique invariantes estruturais (ponteiros consistentes, tamanho correto) após cada sequência e descreva a complexidade por operação.

*Tarefa 1: Caminhos Mínimos com Arestas Negativas — Algoritmo de Bellman-Ford*

In [ ]:
def bellman_ford(num_vertices, edges, start_node):
    """
    Algoritmo de Bellman-Ford para caminhos mínimos.
    Suporta arestas com pesos negativos e deteta ciclos de peso negativo.
    - edges: lista de tuplas (u, v, peso)
    Complexidade Temporal: O(V * E)
    """
    distances = [float('inf')] * num_vertices
    predecessors = [None] * num_vertices
    distances[start_node] = 0

    # Relaxamento de todas as arestas (V - 1) vezes
    for _ in range(num_vertices - 1):
        for u, v, weight in edges:
            if distances[u] != float('inf') and distances[u] + weight < distances[v]:
                distances[v] = distances[u] + weight
                predecessors[v] = u

    # 1.b: Verificação de ciclo negativo na V-ésima passagem
    has_negative_cycle = False
    for u, v, weight in edges:
        if distances[u] != float('inf') and distances[u] + weight < distances[v]:
            has_negative_cycle = True
            break

    return distances, predecessors, has_negative_cycle


# Demonstração
num_v = 5
edges_with_neg = [
    (0, 1, -1), (0, 2, 4), (1, 2, 3),
    (1, 3, 2), (1, 4, 2), (3, 2, 5),
    (3, 1, 1), (4, 3, -3)
]

dist_bf, pred_bf, neg_cycle = bellman_ford(num_v, edges_with_neg, start_node=0)

print("--- Algoritmo de Bellman-Ford ---")
print(f"Distâncias a partir do nó 0: {dist_bf}")
print(f"Contém ciclo de peso negativo? {neg_cycle}")

--- Algoritmo de Bellman-Ford ---
Distâncias a partir do nó 0: [0, -1, 2, -2, 1]
Contém ciclo de peso negativo? False


*Comparação Técnica: Dijkstra vs. Bellman-Ford*
- Por que o Dijkstra falha com pesos negativos? O Dijkstra assume de forma gulosa que, uma vez que um vértice é visitado e marcado como finalizado, a sua menor distância já foi encontrada. Arestas negativas posteriores podem oferecer um caminho mais curto para vértices já processados, invalidando essa premissa.
- Mecanismo do Bellman-Ford: Executa $V - 1$ iterações sobre a totalidade de arestas $E$, garantindo a propagação da menor distância mesmo ao longo de sequências com pesos negativos. Se na $V$-ésima iteração ainda for possível relaxar alguma aresta, confirma-se a presença de um ciclo de peso negativo (onde a distância tende a $-\infty$).

*Tarefa 2: Caminhos Mínimos Par a Par — Algoritmo de Floyd-Warshall*

In [ ]:
def floyd_warshall(num_vertices, adj_matrix):
    """
    Algoritmo de Floyd-Warshall (Programação Dinâmica).
    Calcula as distâncias mínimas entre todos os pares de vértices.
    Complexidade Temporal: O(V^3) | Espacial: O(V^2)
    """
    # Cria cópia da matriz de adjacência original
    dist = [row[:] for row in adj_matrix]

    # Matriz k atua como vértice intermediário
    for k in range(num_vertices):
        for i in range(num_vertices):
            for j in range(num_vertices):
                if dist[i][k] != float('inf') and dist[k][j] != float('inf'):
                    if dist[i][k] + dist[k][j] < dist[i][j]:
                        dist[i][j] = dist[i][k] + dist[k][j]

    return dist


# Demonstração com 4 vértices
inf = float('inf')
graph_matrix = [
    [0,   5,   inf, 10],
    [inf, 0,   3,   inf],
    [inf, inf, 0,   1],
    [inf, inf, inf, 0]
]

all_pairs_dist = floyd_warshall(4, graph_matrix)

print("--- Matriz de Caminhos Mínimos Todos-para-Todos (Floyd-Warshall) ---")
for row in all_pairs_dist:
    print(row)

--- Matriz de Caminhos Mínimos Todos-para-Todos (Floyd-Warshall) ---
[0, 5, 8, 9]
[inf, 0, 3, 4]
[inf, inf, 0, 1]
[inf, inf, inf, 0]


*Análise de Aplicabilidade*
- Floyd-Warshall vs. Repetição de Dijkstra: Executar o Dijkstra $V$ vezes com Min-Heap possui complexidade $O(V \cdot (V + E) \log V) = O(V^2 \log V + V E \log V)$. Para grafos densos ($E \approx V^2$), isso equivale a $O(V^3 \log V)$, tornando o Floyd-Warshall ($O(V^3)$) estruturalmente mais simples, mais rápido em prática devido a baixo overhead de laços e nativamente capaz de lidar com arestas negativas.

*Tarefa 3: Fluxo Máximo em Redes — Algoritmo de Edmonds-Karp (Ford-Fulkerson com BFS)*

In [ ]:
from collections import deque

def edmonds_karp(capacity_matrix, source, sink):
    """
    Algoritmo de Edmonds-Karp para encontrar o Fluxo Máximo em uma rede.
    Utiliza BFS para encontrar o caminho aumentante mais curto em número de arestas.
    Complexidade Temporal: O(V * E^2)
    """
    n = len(capacity_matrix)
    residual_capacity = [row[:] for row in capacity_matrix]
    parent = [-1] * n
    max_flow = 0

    def bfs():
        nonlocal parent
        parent = [-1] * n
        parent[source] = source
        queue = deque([(source, float('inf'))])

        while queue:
            u, flow = queue.popleft()
            if u == sink:
                return flow

            for v in range(n):
                # Se não foi visitado e possui capacidade residual
                if parent[v] == -1 and residual_capacity[u][v] > 0:
                    parent[v] = u
                    new_flow = min(flow, residual_capacity[u][v])
                    if v == sink:
                        return new_flow
                    queue.append((v, new_flow))
        return 0

    # Busca sucessiva por caminhos aumentantes
    while True:
        path_flow = bfs()
        if path_flow == 0:
            break
        max_flow += path_flow

        # Atualiza as capacidades residuais ao longo do caminho encontrado
        v = sink
        while v != source:
            u = parent[v]
            residual_capacity[u][v] -= path_flow
            residual_capacity[v][u] += path_flow  # Aresta de retorno (back-edge)
            v = u

    return max_flow


# Demonstração
# Grafo de Capacidades com 6 vértices (0 = Origem / Source, 5 = Poço / Sink)
capacities = [
    [0, 16, 13, 0,  0,  0],
    [0, 0,  10, 12, 0,  0],
    [0, 4,  0,  0,  14, 0],
    [0, 0,  9,  0,  0,  20],
    [0, 0,  0,  7,  0,  4],
    [0, 0,  0,  0,  0,  0]
]

source_node = 0
sink_node = 5
max_flow_val = edmonds_karp(capacities, source_node, sink_node)

print("--- Fluxo Máximo (Edmonds-Karp) ---")
print(f"Fluxo Máximo do Nó {source_node} até o Nó {sink_node}: {max_flow_val}")

--- Fluxo Máximo (Edmonds-Karp) ---
Fluxo Máximo do Nó 0 até o Nó 5: 23


*Teorema do Fluxo Máximo e Corte Mínimo (Max-Flow Min-Cut)*
- Garantia Teórica: O valor do fluxo máximo enviável da origem ao destino numa rede é exatamente igual à capacidade total acumulada do corte mínimo (Min-Cut), que consiste no conjunto de arestas de capacidade mínima cuja remoção desconecta completamente a fonte do poço.

### Exercício 12: Motor de indexação e consulta com estruturas combinadas e otimização

**Introdução**  
Este exercício consolida competências de engenharia de algoritmos e estruturas de dados em um cenário realista: construir um motor de indexação e consulta para documentos textuais. Em sistemas profissionais, mecanismos desse tipo fundamentam busca interna, observabilidade, auditoria e análise de logs. O desafio não é apenas "funcionar", mas entregar desempenho consistente, operar com estruturas adequadas e justificar tecnicamente as escolhas. Você deverá implementar uma cadeia completa: tokenização, indexação invertida, execução de consultas com operadores e ranqueamento, além de instrumentação e análise de complexidade. O exercício exige integração obrigatória de hashtable, pilha, fila, recursão, programação dinâmica, ordenação eficiente, lista encadeada e árvore. A avaliação considerará corretude, qualidade de testes, documentação técnica e rigor na análise de custos, incluindo identificação de gargalos e otimizações.

**Tarefa**  
1. Indexação invertida com estruturas obrigatórias:
   - a. Receba uma lista de documentos (strings) definida no notebook e tokenize em termos normalizados;
   - b. Construa um índice invertido em uma `HashTableChained` onde cada termo aponta para uma lista encadeada de ocorrências (`doc_id`, posições);
   - c. Mantenha uma BST de termos para permitir listagem ordenada e busca binária em árvore por termo.
2. Consultas booleanas e execução:
   - a. Implemente parsing de consultas com parênteses e operadores AND, OR, NOT usando uma `Stack` para conversão infixa para pós-fixa e avaliação;
   - b. Execute a consulta retornando o conjunto de `doc_ids` correspondentes, usando operações eficientes sobre listas ordenadas quando aplicável;
   - c. Use uma `Queue` para percorrer em largura a BST e gerar um relatório de diagnóstico (níveis, balanceamento aproximado, termos por nível).
3. Ranqueamento, otimização e programação dinâmica:
   - a. Para os `doc_ids` retornados, compute uma pontuação simples baseada em frequência e posição, e ordene os resultados usando QuickSort instrumentado;
   - b. Implemente um componente de sugestão de termo para consultas com termos ausentes, usando programação dinâmica (por exemplo, distância de edição com memoization em hashtable);
   - c. Produza uma análise completa: identifique o custo assintótico de indexação, de consulta, de ranqueamento e do módulo de sugestão, e proponha duas otimizações concretas com justificativa baseada em Big O e em evidências do seu notebook.

*Tarefa 1: Maior Subsequência Comum (LCS — Longest Common Subsequence)*

In [ ]:
def lcs(seq1, seq2):
    """
    Calcula a extensão e reconstrói a Maior Subsequência Comum entre duas sequências.
    Complexidade Temporal: O(m * n) | Complexidade Espacial: O(m * n)
    """
    m, n = len(seq1), len(seq2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Preenchimento da tabela de programação dinâmica
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[i - 1] == seq2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    # Reconstrução retroativa (backtracking na tabela DP)
    lcs_chars = []
    i, j = m, n
    while i > 0 and j > 0:
        if seq1[i - 1] == seq2[j - 1]:
            lcs_chars.append(seq1[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1

    lcs_chars.reverse()
    return dp[m][n], "".join(lcs_chars)


# Demonstração
s1 = "AGGTAB"
s2 = "GXTXAYB"
length_lcs, str_lcs = lcs(s1, s2)

print("--- Maior Subsequência Comum (LCS) ---")
print(f"Sequência 1: {s1}")
print(f"Sequência 2: {s2}")
print(f"Tamanho da LCS: {length_lcs} | Subsequência: '{str_lcs}'")

--- Maior Subsequência Comum (LCS) ---
Sequência 1: AGGTAB
Sequência 2: GXTXAYB
Tamanho da LCS: 4 | Subsequência: 'GTAB'


*Análise de Complexidade e Otimização Espacial*
- Complexidade de Tempo e Espaço: O algoritmo básico exige tempo $O(m \cdot n)$ e espaço $O(m \cdot n)$ para armazenar a matriz de tamanho $(m + 1) \times (n + 1)$.
- Otimização de Memória: Se apenas a extensão da LCS for necessária (sem a reconstrução da cadeia), é possível manter apenas duas linhas da tabela na memória (linha atual e anterior), reduzindo o consumo espacial para $O(\min(m, n))$.

*Tarefa 2: Distância de Edição (Edit Distance / Levenshtein Distance)*

In [ ]:
def min_edit_distance(str1, str2):
    """
    Calcula o número mínimo de operações (Inserção, Deleção, Substituição)
    necessárias para transformar str1 em str2.
    Complexidade Temporal: O(m * n) | Complexidade Espacial: O(m * n)
    """
    m, n = len(str1), len(str2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Casos base: transformar string vazia
    for i in range(m + 1):
        dp[i][0] = i  # i deleções
    for j in range(n + 1):
        dp[0][j] = j  # j inserções

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i - 1] == str2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],     # Custo de Deleção
                    dp[i][j - 1],     # Custo de Inserção
                    dp[i - 1][j - 1]  # Custo de Substituição
                )

    return dp[m][n]


# Demonstração
w1 = "GATO"
w2 = "PRATO"
dist = min_edit_distance(w1, w2)

print("--- Distância de Edição (Levenshtein) ---")
print(f"Transformar '{w1}' em '{w2}': {dist} operações mínimas.")

--- Distância de Edição (Levenshtein) ---
Transformar 'GATO' em 'PRATO': 2 operações mínimas.


*Aplicações Práticas da Distância de Edição*
- Bioinformática: Alinhamento de sequências de DNA e RNA para identificar similaridades genéticas entre espécies.
- Processamento de Linguagem Natural (PLN): Corretores ortográficos automáticos, sugestões de autocompletar e verificação de plágio em documentos.

*Tarefa 3: Multiplicação de Cadeia de Matrizes (Matrix Chain Multiplication - MCM)*

In [ ]:
def matrix_chain_order(p):
    """
    Determina a parentetização ótima para multiplicar uma cadeia de matrizes,
    minimizando o número total de multiplicações escalares.
    - p: lista de dimensões. A matriz Ai possui dimensão p[i-1] x p[i].
    Complexidade Temporal: O(n^3) | Complexidade Espacial: O(n^2)
    """
    n = len(p) - 1  # Número de matrizes
    m = [[0] * n for _ in range(n)]
    s = [[0] * n for _ in range(n)]

    # l é o comprimento da subcadeia de matrizes
    for l in range(2, n + 1):
        for i in range(n - l + 1):
            j = i + l - 1
            m[i][j] = float('inf')
            for k in range(i, j):
                # Custo = custo(A[i..k]) + custo(A[k+1..j]) + custo de multiplicar as duas resultantes
                q = m[i][k] + m[k + 1][j] + p[i] * p[k + 1] * p[j + 1]
                if q < m[i][j]:
                    m[i][j] = q
                    s[i][j] = k

    return m[0][n - 1], s


def format_parenthesization(s, i, j):
    if i == j:
        return f"A{i + 1}"
    else:
        left = format_parenthesization(s, i, s[i][j])
        right = format_parenthesization(s, s[i][j] + 1, j)
        return f"({left} x {right})"


# Demonstração
# 4 Matrizes: A1 (10x30), A2 (30x5), A3 (5x60), A4 (60x8)
dimensions = [10, 30, 5, 60, 8]

min_ops, split_table = matrix_chain_order(dimensions)
optimal_structure = format_parenthesization(split_table, 0, len(dimensions) - 2)

print("--- Multiplicação de Cadeia de Matrizes ---")
print(f"Número mínimo de multiplicações escalares: {min_ops}")
print(f"Ordem ótima de parentetização: {optimal_structure}")

--- Multiplicação de Cadeia de Matrizes ---
Número mínimo de multiplicações escalares: 4300
Ordem ótima de parentetização: ((A1 x A2) x (A3 x A4))


*Impacto da Ordem de Associação nas Operações*
- Importância do Agrupamento Ótimo: A multiplicação de matrizes é associativa ($A \cdot (B \cdot C) = (A \cdot B) \cdot C$), mas a quantidade de multiplicações escalares varia drasticamente dependendo da ordem escolhida.
- Exemplo Prático: Para matrizes com dimensões $10 \times 30$, $30 \times 5$ e $5 \times 60$:
  - $(A_1 \cdot A_2) \cdot A_3$ efetua $(10 \times 30 \times 5) + (10 \times 5 \times 60) = 1500 + 3000 = 4500$ operações.
  - $A_1 \cdot (A_2 \cdot A_3)$ efetua $(30 \times 5 \times 60) + (10 \times 30 \times 60) = 9000 + 18000 = 27000$ operações.
  - O algoritmo de Programação Dinâmica encontra a primeira ordem em tempo $O(n^3)$, evitando testar todas as combinações possíveis que crescem exponencialmente segundo os Números de Catalan.